# Traffic Demand Prediction — v2 (Optuna LGB + CatBoost Ensemble)

No MIGA — simple imputation. Optuna tunes LGB hyperparams on a fast temporal split, then both LGB and CatBoost are trained with 5-fold CV and averaged.

In [1]:
# !pip install lightgbm catboost optuna scikit-learn pandas numpy

In [2]:
import sys, warnings, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import lightgbm as lgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

_nb_dir = Path('.').resolve()
DATA_DIR = next(
    (p for p in [_nb_dir/'data', _nb_dir/'dataset', Path('data'), Path('dataset')] if p.exists()),
    Path('data')
)

SEED = 42
np.random.seed(SEED)
N_FOLDS     = 5
N_OPT_TRIALS = 40
USE_LOG1P   = False  # log1p tested — hurt OOF by -0.0004, reverted
t0 = time.time()
print(f'Data dir: {DATA_DIR}')

Data dir: /home/kesav/Projects/gridlock-hackathon/hackathon submission/data


/home/kesav/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Geohash decoder

In [3]:
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'

def decode_geohash(gh: str):
    lat_r = [-90.0, 90.0]; lon_r = [-180.0, 180.0]; is_lon = True
    for c in gh:
        cd = _BASE32.index(c)
        for bits in [16, 8, 4, 2, 1]:
            if is_lon:
                mid = (lon_r[0] + lon_r[1]) / 2
                if cd & bits: lon_r[0] = mid
                else:         lon_r[1] = mid
            else:
                mid = (lat_r[0] + lat_r[1]) / 2
                if cd & bits: lat_r[0] = mid
                else:         lat_r[1] = mid
            is_lon = not is_lon
    return (lat_r[0] + lat_r[1]) / 2, (lon_r[0] + lon_r[1]) / 2
def encode_geohash(lat, lon, precision=6):
    lat_r = [-90.0, 90.0]; lon_r = [-180.0, 180.0]
    is_lon = True; bits = 0; bit = 0; result = ''
    while len(result) < precision:
        if is_lon:
            mid = (lon_r[0]+lon_r[1])/2
            if lon >= mid: lon_r[0] = mid; bit = (bit << 1) | 1
            else: lon_r[1] = mid; bit = bit << 1
        else:
            mid = (lat_r[0]+lat_r[1])/2
            if lat >= mid: lat_r[0] = mid; bit = (bit << 1) | 1
            else: lat_r[1] = mid; bit = bit << 1
        is_lon = not is_lon; bits += 1
        if bits == 5:
            result += _BASE32[bit]; bits = 0; bit = 0
    return result


## Load data & feature engineering

In [4]:
train_raw = pd.read_csv(DATA_DIR / 'train.csv')
test_raw  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'train: {train_raw.shape}   test: {test_raw.shape}')

def add_time_features(df):
    parts = df['timestamp'].str.split(':', expand=True).astype(float)
    df = df.copy()
    df['hour']       = parts[0].values
    df['minute']     = parts[1].values
    df['ts_minutes'] = df['hour'] * 60 + df['minute']
    df['time_slot']  = (df['ts_minutes'] // 15).astype(int)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin']    = np.sin(2 * np.pi * df['day'] / 7)
    df['day_cos']    = np.cos(2 * np.pi * df['day'] / 7)
    return df

train_raw = add_time_features(train_raw)
test_raw  = add_time_features(test_raw)

_gh_coords = {gh: decode_geohash(gh)
              for gh in pd.concat([train_raw['geohash'], test_raw['geohash']]).unique()}

for df in [train_raw, test_raw]:
    df['lat'] = df['geohash'].map(lambda g: _gh_coords[g][0])
    df['lon'] = df['geohash'].map(lambda g: _gh_coords[g][1])
    df['gh4'] = df['geohash'].str[:4]
    df['gh5'] = df['geohash'].str[:5]

# demand_lag: day48 demand for same (geohash, timestamp) — strongest feature
lag_d48 = (train_raw[train_raw['day']==48][['geohash','timestamp','demand']]
           .rename(columns={'demand':'demand_lag'}))
train_raw = train_raw.merge(lag_d48, on=['geohash','timestamp'], how='left')
train_raw['demand_lag'] = np.where(train_raw['day']==48, np.nan, train_raw['demand_lag'])
test_raw = test_raw.merge(lag_d48, on=['geohash','timestamp'], how='left')

# geo_day_trend: is day49 busier/quieter than day48 per geohash?
_d48g = train_raw[train_raw['day']==48].groupby('geohash')['demand'].mean()
_d49g = train_raw[train_raw['day']==49].groupby('geohash')['demand'].mean()
_trend = (_d49g / (_d48g + 1e-6))
_gtrend = float(_trend.mean())
for df in [train_raw, test_raw]:
    df['geo_day_trend'] = df['geohash'].map(_trend).fillna(_gtrend)

print(f"demand_lag coverage — train day49: {train_raw.loc[train_raw['day']==49,'demand_lag'].notna().mean()*100:.1f}%  "
      f"test: {test_raw['demand_lag'].notna().mean()*100:.1f}%")
print(f"geo_day_trend — mean={_gtrend:.3f}  min={_trend.min():.3f}  max={_trend.max():.3f}")


# Neighbor geohash mean demand — spatial smoothing feature
_geo_demand_mean = train_raw.groupby('geohash')['demand'].mean()
_global_demand_mean = float(_geo_demand_mean.mean())

def _get_neighbor_mean(gh):
    lat, lon = _gh_coords[gh]
    lat_r = [-90.0, 90.0]; lon_r = [-180.0, 180.0]; is_lon = True
    for ch in gh:
        cd = _BASE32.index(ch)
        for b in [16, 8, 4, 2, 1]:
            if is_lon:
                mid = (lon_r[0]+lon_r[1])/2
                if cd & b: lon_r[0] = mid
                else: lon_r[1] = mid
            else:
                mid = (lat_r[0]+lat_r[1])/2
                if cd & b: lat_r[0] = mid
                else: lat_r[1] = mid
            is_lon = not is_lon
    lat_step = lat_r[1] - lat_r[0]; lon_step = lon_r[1] - lon_r[0]
    demands = [_geo_demand_mean[encode_geohash(lat + dlat*lat_step, lon + dlon*lon_step, len(gh))]
               for dlat in [-1,0,1] for dlon in [-1,0,1]
               if not (dlat==0 and dlon==0)
               and encode_geohash(lat + dlat*lat_step, lon + dlon*lon_step, len(gh)) in _geo_demand_mean]
    return float(np.mean(demands)) if demands else _global_demand_mean

_gh_neighbor_mean = {gh: _get_neighbor_mean(gh) for gh in _gh_coords}
for df in [train_raw, test_raw]:
    df['neighbor_demand_mean'] = df['geohash'].map(_gh_neighbor_mean).fillna(_global_demand_mean)
print(f"neighbor_demand_mean — mean={train_raw['neighbor_demand_mean'].mean():.4f}  "
      f"corr_with_demand={train_raw['neighbor_demand_mean'].corr(train_raw['demand']):.4f}")

# Better demand_lag fallback: geohash d48 mean instead of global median for 11.1% missing test rows
test_raw['demand_lag'] = test_raw['demand_lag'].fillna(
    test_raw['geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
)
print(f"demand_lag after geo fallback — test NaN remaining: "
      f"{test_raw['demand_lag'].isna().sum()} (was 4642)")


# ── D48 Temporal Profile Features ──────────────────────────────────────────
# For each geohash, use the full day-48 demand curve to build richer features.
# d48 has all 96 timestamps — far more info than just demand_lag (1 point).

_test_ts_set = set(test_raw['timestamp'].unique())  # 47 daytime timestamps 2:15-13:45

# 1. Per-geohash d48 daytime stats (mean/std/max over test-equivalent timestamps)
_d48_daytime = train_raw[
    (train_raw['day'] == 48) & train_raw['timestamp'].isin(_test_ts_set)
]
_gh_d48dt_mean = _d48_daytime.groupby('geohash')['demand'].mean()
_gh_d48dt_std  = _d48_daytime.groupby('geohash')['demand'].std().fillna(0)
_gh_d48dt_max  = _d48_daytime.groupby('geohash')['demand'].max()
_global_d48dt_mean = float(_gh_d48dt_mean.mean())

for df in [train_raw, test_raw]:
    df['d48_daytime_mean'] = df['geohash'].map(_gh_d48dt_mean).fillna(_global_d48dt_mean)
    df['d48_daytime_std']  = df['geohash'].map(_gh_d48dt_std).fillna(0)
    df['d48_daytime_max']  = df['geohash'].map(_gh_d48dt_max).fillna(_global_d48dt_mean)

# 2. Per-geohash d48 pivot table (geohash × time_slot demand)
_d48_pivot = (train_raw[train_raw['day'] == 48]
              .groupby(['geohash', 'time_slot'])['demand'].mean()
              .unstack(fill_value=np.nan))  # shape: n_geohashes × 96

_all_slots = list(range(96))

def _d48_slot(gh, slot):
    """Fetch d48 demand for (geohash, time_slot), NaN if missing."""
    if gh not in _d48_pivot.index or slot < 0 or slot >= 96:
        return np.nan
    v = _d48_pivot.loc[gh, slot] if slot in _d48_pivot.columns else np.nan
    return float(v) if not (isinstance(v, float) and np.isnan(v)) else np.nan

def _d48_interp(gh, slot):
    """Linear interpolation within d48 profile for missing slots."""
    if gh not in _d48_pivot.index:
        return np.nan
    row = _d48_pivot.loc[gh].dropna()
    if len(row) == 0:
        return np.nan
    if slot in row.index:
        return float(row[slot])
    idxs = row.index.values
    dists = np.abs(idxs - slot)
    nearest = idxs[np.argsort(dists)[:2]]
    if len(nearest) < 2:
        return float(row[nearest[0]])
    t1, t2 = nearest[0], nearest[1]
    w1 = 1.0 / (abs(t1 - slot) + 1e-6)
    w2 = 1.0 / (abs(t2 - slot) + 1e-6)
    return float((w1 * row[t1] + w2 * row[t2]) / (w1 + w2))

# 3. Improved demand_lag for missing test rows: linear interpolation within d48 profile
_test_missing = test_raw['demand_lag'].isna()
if _test_missing.sum() > 0:
    interp_vals = [_d48_interp(gh, ts)
                   for gh, ts in zip(test_raw.loc[_test_missing, 'geohash'],
                                     test_raw.loc[_test_missing, 'time_slot'])]
    test_raw.loc[_test_missing, 'demand_lag'] = interp_vals
    # Remaining NaN → geohash mean (cold-start)
    test_raw['demand_lag'] = test_raw['demand_lag'].fillna(
        test_raw['geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
    )
    print(f"demand_lag interpolation: filled {_test_missing.sum()} missing rows "
          f"({test_raw['demand_lag'].isna().sum()} NaN remaining)")

# 4. Masked window features — NaN for day-48 rows, valid for day-49 + test only
# Day-48 rows have within-day adjacent slots (same-day autocorrelation = leaky).
# Day-49 rows + test have cross-day adjacent slots from d48 = legitimate signal.
print('Building masked d48 window features...')
for df in [train_raw, test_raw]:
    ghs   = df['geohash'].values
    slots = df['time_slot'].values
    lag1  = np.array([_d48_slot(g, s-1) for g, s in zip(ghs, slots)])
    lag2  = np.array([_d48_slot(g, s-2) for g, s in zip(ghs, slots)])
    lead1 = np.array([_d48_slot(g, s+1) for g, s in zip(ghs, slots)])
    lead2 = np.array([_d48_slot(g, s+2) for g, s in zip(ghs, slots)])
    win   = np.array([np.nanmean([_d48_slot(g, s+d) for d in range(-4, 5)])
                      for g, s in zip(ghs, slots)])
    df['d48_lag1']        = lag1
    df['d48_lag2']        = lag2
    df['d48_lead1']       = lead1
    df['d48_lead2']       = lead2
    df['d48_window_mean'] = win

# Mask to NaN for ALL day-48 training rows
_d48_row_mask = train_raw['day'] == 48
for col in ['d48_lag1', 'd48_lag2', 'd48_lead1', 'd48_lead2', 'd48_window_mean']:
    train_raw.loc[_d48_row_mask, col] = np.nan
print(f"Masked window features: day-49 coverage={train_raw.loc[~_d48_row_mask,'d48_lag1'].notna().mean():.1%}  "
      f"test coverage={test_raw['d48_lag1'].notna().mean():.1%}")

# 5. Interaction features
for df in [train_raw, test_raw]:
    df['lag_x_trend']    = df['demand_lag'] * df['geo_day_trend']
    df['lag_x_hour_sin'] = df['demand_lag'] * df['hour_sin']
    df['lag_normalized'] = df['demand_lag'] / (df['d48_daytime_mean'] + 1e-6)

print(f"d48_daytime_mean corr with demand: {train_raw['d48_daytime_mean'].corr(train_raw['demand']):.4f}")
print(f"d48_window_mean  corr with demand: {train_raw['d48_window_mean'].corr(train_raw['demand']):.4f}")
print(f"lag_x_trend      corr with demand: {train_raw['lag_x_trend'].corr(train_raw['demand']):.4f}")
print(f"lag_normalized   corr with demand: {train_raw['lag_normalized'].corr(train_raw['demand']):.4f}")

y_train = train_raw['demand'].values
n_train = len(train_raw)

train: (77299, 11)   test: (41778, 10)
demand_lag coverage — train day49: 81.6%  test: 88.9%
geo_day_trend — mean=1.270  min=0.000  max=7.249
neighbor_demand_mean — mean=0.0818  corr_with_demand=0.5220
demand_lag after geo fallback — test NaN remaining: 0 (was 4642)


## Imputation

Simple mode/mean imputation for LGB. CatBoost receives raw data (handles NaN/missing natively).

In [5]:
CAT_COLS = ['geohash', 'gh4', 'gh5', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
NUM_COLS = ['day', 'hour', 'minute', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
            'NumberofLanes', 'Temperature', 'lat', 'lon', 'geo_day_trend', 'neighbor_demand_mean',
            'd48_daytime_mean', 'd48_daytime_std', 'd48_daytime_max',
            'd48_lag1', 'd48_lag2', 'd48_lead1', 'd48_lead2', 'd48_window_mean',
            'lag_x_trend', 'lag_x_hour_sin', 'lag_normalized']

# --- LGB version: label-encode + mode/mean impute ---
train_lgb = train_raw.copy()
test_lgb  = test_raw.copy()

for col in ['RoadType', 'Weather']:
    mode_val = train_lgb[col].mode()[0]
    train_lgb[col] = train_lgb[col].fillna(mode_val)
    test_lgb[col]  = test_lgb[col].fillna(mode_val)

temp_mean = train_lgb['Temperature'].mean()
train_lgb['Temperature'] = train_lgb['Temperature'].fillna(temp_mean)
test_lgb['Temperature']  = test_lgb['Temperature'].fillna(temp_mean)

encoders = {}
combined_cat = pd.concat([train_lgb[CAT_COLS], test_lgb[CAT_COLS]], ignore_index=True)
for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(combined_cat[col].dropna().unique())
    encoders[col] = le
    train_lgb[col] = le.transform(train_lgb[col].astype(str))
    test_lgb[col]  = le.transform(test_lgb[col].astype(str))

# --- CatBoost version: fill categorical NaN with 'Missing', keep strings ---
train_cat = train_raw.copy()
test_cat  = test_raw.copy()

for col in CAT_COLS:
    train_cat[col] = train_cat[col].fillna('Missing').astype(str)
    test_cat[col]  = test_cat[col].fillna('Missing').astype(str)

# Temperature NaN handled natively by CatBoost

print(f"Missing after LGB imputation — train: {train_lgb[CAT_COLS+NUM_COLS].isna().sum().sum()}  "
      f"test: {test_lgb[CAT_COLS+NUM_COLS].isna().sum().sum()}")
print(f"CatBoost categorical NaN replaced with 'Missing'")

Missing after LGB imputation — train: 0  test: 0
CatBoost categorical NaN replaced with 'Missing'


## OOF target encoding (for LGB)

Geohash×hour, gh5×hour, geohash, RoadType×hour, geohash×time_slot, gh5×time_slot — all out-of-fold to prevent leakage.

In [6]:
def oof_target_encode(train_df, test_df, group_cols, y, kf, smoothing=10.0):
    global_mean = y.mean()
    train_enc   = np.full(len(train_df), global_mean)
    tmp = train_df[group_cols].copy()
    tmp['__y'] = y
    for tr_idx, val_idx in kf.split(train_df):
        stats = (tmp.iloc[tr_idx].groupby(group_cols)['__y']
                 .agg(['mean','count']).reset_index()
                 .rename(columns={'mean':'__m','count':'__n'}))
        stats['__b'] = (stats['__m']*stats['__n'] + global_mean*smoothing) / (stats['__n']+smoothing)
        merged = tmp.iloc[val_idx][group_cols].merge(stats[group_cols+['__b']], on=group_cols, how='left')
        train_enc[val_idx] = merged['__b'].fillna(global_mean).values
    full = (tmp.groupby(group_cols)['__y']
            .agg(['mean','count']).reset_index()
            .rename(columns={'mean':'__m','count':'__n'}))
    full['__b'] = (full['__m']*full['__n'] + global_mean*smoothing) / (full['__n']+smoothing)
    test_enc = (test_df[group_cols]
                .merge(full[group_cols+['__b']], on=group_cols, how='left')['__b']
                .fillna(global_mean).values)
    return train_enc, test_enc

# Add time_slot to lgb frames (needed for geo×time_slot encoding)
train_lgb['time_slot'] = train_raw['time_slot'].values
test_lgb['time_slot']  = test_raw['time_slot'].values

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print('Computing OOF encodings...')
tr_gh_hr,  te_gh_hr  = oof_target_encode(train_lgb, test_lgb, ['geohash','hour'],     y_train, kf)
tr_gh5_hr, te_gh5_hr = oof_target_encode(train_lgb, test_lgb, ['gh5','hour'],          y_train, kf)
tr_gh,     te_gh     = oof_target_encode(train_lgb, test_lgb, ['geohash'],              y_train, kf)
tr_rt_hr,  te_rt_hr  = oof_target_encode(train_lgb, test_lgb, ['RoadType','hour'],     y_train, kf)
tr_geo_ts, te_geo_ts = oof_target_encode(train_lgb, test_lgb, ['geohash','time_slot'], y_train, kf, smoothing=5.0)
tr_gh5_ts, te_gh5_ts = oof_target_encode(train_lgb, test_lgb, ['gh5','time_slot'],     y_train, kf, smoothing=5.0)
print('Done.')

LGB_FEATURE_COLS = CAT_COLS + NUM_COLS

def build_X_lgb(df, e1, e2, e3, e4, e5, e6):
    out = df[LGB_FEATURE_COLS].copy().values.astype(float)
    lag_med = df['demand_lag'].median()
    lag = df['demand_lag'].fillna(lag_med).values
    return np.column_stack([out, lag, e1, e2, e3, e4, e5, e6])

X_lgb_tr = build_X_lgb(train_lgb, tr_gh_hr, tr_gh5_hr, tr_gh, tr_rt_hr, tr_geo_ts, tr_gh5_ts)
X_lgb_te = build_X_lgb(test_lgb,  te_gh_hr, te_gh5_hr, te_gh, te_rt_hr, te_geo_ts, te_gh5_ts)
print(f'LGB feature matrix: train {X_lgb_tr.shape}   test {X_lgb_te.shape}')

Computing OOF encodings...
Done.
LGB feature matrix: train (77299, 27)   test (41778, 27)


## LGB hyperparameters

Proven params from MIGA run. Optuna search on temporal split found over-regularized values because `demand_lag=NaN` for all day48 rows there — those params don't transfer to the full 5-fold CV where day49 rows have demand_lag populated.

In [7]:
# LGB hyperparameters — proven params from MIGA run (OOF R²=0.9543)
# Optuna on temporal split found over-regularized params (demand_lag=NaN for all
# day48 training rows → model can't use it → needs heavy regularization there,
# but in full CV day49 rows have demand_lag → lighter regularization is better)

best_lgb_params = dict(
    objective='regression', metric='rmse',
    n_estimators=3000, learning_rate=0.02,
    num_leaves=127, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, verbose=-1, random_state=SEED,
)

print("LGB params (proven from MIGA run):")
for k, v in best_lgb_params.items():
    if k not in ('objective','metric','n_jobs','verbose','random_state'):
        print(f"  {k}: {v}")


LGB params (proven from MIGA run):
  n_estimators: 3000
  learning_rate: 0.02
  num_leaves: 127
  min_child_samples: 20
  subsample: 0.8
  colsample_bytree: 0.8
  reg_alpha: 0.1
  reg_lambda: 0.1


## LightGBM — 5-fold CV with Optuna-tuned params

In [8]:
oof_lgb  = np.zeros(n_train)
test_lgb_preds = np.zeros(len(test_raw))
fold_scores_lgb = []

print('Training LGB...')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_lgb_tr)):
    y_tr = np.log1p(y_train[tr_idx]) if USE_LOG1P else y_train[tr_idx]
    y_va = y_train[val_idx]

    model = lgb.LGBMRegressor(**best_lgb_params)
    model.fit(X_lgb_tr[tr_idx], y_tr,
              eval_set=[(X_lgb_tr[val_idx],
                         np.log1p(y_va) if USE_LOG1P else y_va)],
              callbacks=[lgb.early_stopping(150, verbose=False),
                         lgb.log_evaluation(500)])

    raw = model.predict(X_lgb_tr[val_idx])
    oof_lgb[val_idx] = np.clip(np.expm1(raw) if USE_LOG1P else raw, 0, 1)

    raw_te = model.predict(X_lgb_te)
    test_lgb_preds += np.clip(np.expm1(raw_te) if USE_LOG1P else raw_te, 0, 1) / N_FOLDS

    fs = r2_score(y_va, oof_lgb[val_idx])
    rmse = np.sqrt(np.mean((y_va - oof_lgb[val_idx])**2))
    fold_scores_lgb.append(fs)
    print(f'  Fold {fold+1}  R²={fs:.4f}  RMSE={rmse:.5f}  iter={model.best_iteration_}')

lgb_oof_r2 = r2_score(y_train, oof_lgb)
print(f'\nLGB OOF R² = {lgb_oof_r2:.4f}  →  {max(0, 100*lgb_oof_r2):.2f}/100')

Training LGB...
[500]	valid_0's rmse: 0.0312035
[1000]	valid_0's rmse: 0.0310328
[1500]	valid_0's rmse: 0.0310014
  Fold 1  R²=0.9526  RMSE=0.03096  iter=1718
[500]	valid_0's rmse: 0.0318538
[1000]	valid_0's rmse: 0.0316083
[1500]	valid_0's rmse: 0.031499
[2000]	valid_0's rmse: 0.0314577
  Fold 2  R²=0.9530  RMSE=0.03143  iter=2132
[500]	valid_0's rmse: 0.0299957
[1000]	valid_0's rmse: 0.0297312
[1500]	valid_0's rmse: 0.0297064
  Fold 3  R²=0.9563  RMSE=0.02968  iter=1365
[500]	valid_0's rmse: 0.0305651
[1000]	valid_0's rmse: 0.0303144
  Fold 4  R²=0.9523  RMSE=0.03025  iter=1320
[500]	valid_0's rmse: 0.0305729
[1000]	valid_0's rmse: 0.0302447
[1500]	valid_0's rmse: 0.0301861
[2000]	valid_0's rmse: 0.0301446
[2500]	valid_0's rmse: 0.0301128
  Fold 5  R²=0.9558  RMSE=0.03009  iter=2544

LGB OOF R² = 0.9540  →  95.40/100


## CatBoost + XGBoost — 5-fold CV

CatBoost: native categorical OTS on raw string columns. XGBoost: depth-wise trees on the same LGB feature matrix (manual OOF encodings). Different architectures → different errors → better ensemble.

In [9]:
from xgboost import XGBRegressor

CAT_FEAT_COLS = CAT_COLS + NUM_COLS + ['time_slot', 'demand_lag']
cat_feature_indices = list(range(len(CAT_COLS)))

train_cat['time_slot']  = train_raw['time_slot'].values
test_cat['time_slot']   = test_raw['time_slot'].values
train_cat['demand_lag'] = train_raw['demand_lag'].values
test_cat['demand_lag']  = test_raw['demand_lag'].values

# ── CatBoost ─────────────────────────────────────────────────────────────────
oof_cat  = np.zeros(n_train)
test_cat_preds = np.zeros(len(test_raw))
fold_scores_cat = []

catboost_params = dict(
    iterations=8000, learning_rate=0.03, depth=8,
    loss_function='RMSE', random_seed=SEED, verbose=0,
    early_stopping_rounds=200,
)

print('Training CatBoost (depth=8, lr=0.03)...')
for fold, (tr_idx, val_idx) in enumerate(kf.split(train_cat)):
    y_tr = np.log1p(y_train[tr_idx]) if USE_LOG1P else y_train[tr_idx]
    y_va = y_train[val_idx]

    cb = CatBoostRegressor(**catboost_params)
    cb.fit(
        train_cat.iloc[tr_idx][CAT_FEAT_COLS], y_tr,
        cat_features=cat_feature_indices,
        eval_set=(train_cat.iloc[val_idx][CAT_FEAT_COLS],
                  np.log1p(y_va) if USE_LOG1P else y_va),
        use_best_model=True
    )

    raw = cb.predict(train_cat.iloc[val_idx][CAT_FEAT_COLS])
    oof_cat[val_idx] = np.clip(np.expm1(raw) if USE_LOG1P else raw, 0, 1)

    raw_te = cb.predict(test_cat[CAT_FEAT_COLS])
    test_cat_preds += np.clip(np.expm1(raw_te) if USE_LOG1P else raw_te, 0, 1) / N_FOLDS

    fs = r2_score(y_va, oof_cat[val_idx])
    fold_scores_cat.append(fs)
    print(f'  Fold {fold+1}  R²={fs:.4f}  RMSE={np.sqrt(np.mean((y_va-oof_cat[val_idx])**2)):.5f}  iter={cb.best_iteration_}')

cat_oof_r2 = r2_score(y_train, oof_cat)
print(f'\nCatBoost OOF R² = {cat_oof_r2:.4f}  →  {max(0, 100*cat_oof_r2):.2f}/100')

# ── XGBoost ───────────────────────────────────────────────────────────────────
oof_xgb  = np.zeros(n_train)
test_xgb_preds = np.zeros(len(test_raw))
fold_scores_xgb = []

# early_stopping_rounds goes in the constructor in XGBoost 2.0+
xgb_params = dict(
    objective='reg:squarederror',
    n_estimators=3000, learning_rate=0.02,
    max_depth=6, min_child_weight=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    early_stopping_rounds=150,
    n_jobs=-1, random_state=SEED, verbosity=0,
)

print('\nTraining XGBoost (max_depth=6, lr=0.02)...')
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_lgb_tr)):
    y_tr = np.log1p(y_train[tr_idx]) if USE_LOG1P else y_train[tr_idx]
    y_va = y_train[val_idx]

    xgb = XGBRegressor(**xgb_params)
    xgb.fit(X_lgb_tr[tr_idx], y_tr,
            eval_set=[(X_lgb_tr[val_idx], np.log1p(y_va) if USE_LOG1P else y_va)],
            verbose=False)

    raw = xgb.predict(X_lgb_tr[val_idx])
    oof_xgb[val_idx] = np.clip(np.expm1(raw) if USE_LOG1P else raw, 0, 1)

    raw_te = xgb.predict(X_lgb_te)
    test_xgb_preds += np.clip(np.expm1(raw_te) if USE_LOG1P else raw_te, 0, 1) / N_FOLDS

    fs = r2_score(y_va, oof_xgb[val_idx])
    fold_scores_xgb.append(fs)
    print(f'  Fold {fold+1}  R²={fs:.4f}  RMSE={np.sqrt(np.mean((y_va-oof_xgb[val_idx])**2)):.5f}  iter={xgb.best_iteration}')

xgb_oof_r2 = r2_score(y_train, oof_xgb)
print(f'\nXGBoost OOF R² = {xgb_oof_r2:.4f}  →  {max(0, 100*xgb_oof_r2):.2f}/100')

Training CatBoost (depth=8, lr=0.03)...
  Fold 1  R²=0.9591  RMSE=0.02878  iter=7993
  Fold 2  R²=0.9589  RMSE=0.02940  iter=7997
  Fold 3  R²=0.9601  RMSE=0.02835  iter=7999
  Fold 4  R²=0.9548  RMSE=0.02943  iter=6887
  Fold 5  R²=0.9603  RMSE=0.02854  iter=7968

CatBoost OOF R² = 0.9587  →  95.87/100

Training XGBoost (max_depth=6, lr=0.02)...
  Fold 1  R²=0.9538  RMSE=0.03056  iter=2899
  Fold 2  R²=0.9542  RMSE=0.03103  iter=2968
  Fold 3  R²=0.9570  RMSE=0.02943  iter=2995
  Fold 4  R²=0.9522  RMSE=0.03028  iter=2838
  Fold 5  R²=0.9543  RMSE=0.03060  iter=2996

XGBoost OOF R² = 0.9543  →  95.43/100


## Ensemble — find optimal blend weight from OOF predictions

In [10]:
# 3-way grid search: find w_lgb, w_xgb so that w_cat = 1 - w_lgb - w_xgb
best_r2, best_wl, best_wx = 0, 0.2, 0.1

for wl in np.arange(0.05, 0.65, 0.05):
    for wx in np.arange(0.05, 0.65, 0.05):
        if wl + wx >= 0.95:
            continue
        wc = 1.0 - wl - wx
        blend = wl * oof_lgb + wx * oof_xgb + wc * oof_cat
        r2 = r2_score(y_train, np.clip(blend, 0, 1))
        if r2 > best_r2:
            best_r2, best_wl, best_wx = r2, wl, wx

best_wc = 1.0 - best_wl - best_wx

print(f'Optimal weights — LGB: {best_wl:.2f}  XGB: {best_wx:.2f}  CatBoost: {best_wc:.2f}')
print(f'\nOOF R² comparison:')
print(f'  LGB alone   : {lgb_oof_r2:.4f}  →  {max(0, 100*lgb_oof_r2):.2f}/100')
print(f'  XGBoost     : {xgb_oof_r2:.4f}  →  {max(0, 100*xgb_oof_r2):.2f}/100')
print(f'  CatBoost    : {cat_oof_r2:.4f}  →  {max(0, 100*cat_oof_r2):.2f}/100')
print(f'  Ensemble    : {best_r2:.4f}  →  {max(0, 100*best_r2):.2f}/100')

test_ensemble = np.clip(
    best_wl * test_lgb_preds + best_wx * test_xgb_preds + best_wc * test_cat_preds,
    0, 1
)
oof_ensemble = np.clip(
    best_wl * oof_lgb + best_wx * oof_xgb + best_wc * oof_cat,
    0, 1
)
final_r2 = r2_score(y_train, oof_ensemble)
print(f'\nFinal ensemble OOF R² = {final_r2:.4f}  →  score ≈ {max(0, 100*final_r2):.2f}/100')
print(f'Total wall time: {(time.time()-t0)/60:.1f} min')

Optimal weights — LGB: 0.20  XGB: 0.15  CatBoost: 0.65

OOF R² comparison:
  LGB alone   : 0.9540  →  95.40/100
  XGBoost     : 0.9543  →  95.43/100
  CatBoost    : 0.9587  →  95.87/100
  Ensemble    : 0.9606  →  96.06/100

Final ensemble OOF R² = 0.9606  →  score ≈ 96.06/100
Total wall time: 11.1 min


## Results logger

In [11]:
_LOG_FILE = Path('results_log.json')
_log = json.load(open(_LOG_FILE)) if _LOG_FILE.exists() else []

_run = {
    "timestamp":         __import__('datetime').datetime.now().strftime("%Y-%m-%d %H:%M"),
    "oof_r2":            round(float(final_r2), 6),
    "competition_score": round(float(max(0, 100 * final_r2)), 4),
    "lb_score":          None,
    "model":             "LGB+XGB+CatBoost ensemble (no MIGA)",
    "lgb_oof_r2":        round(float(lgb_oof_r2), 6),
    "xgb_oof_r2":        round(float(xgb_oof_r2), 6),
    "cat_oof_r2":        round(float(cat_oof_r2), 6),
    "lgb_weight":        round(float(best_wl), 2),
    "xgb_weight":        round(float(best_wx), 2),
    "cat_weight":        round(float(best_wc), 2),
    "lgb_params":        {k: v for k, v in best_lgb_params.items()
                          if k not in ('objective','metric','n_jobs','verbose','random_state')},
    "xgb_params":        {k: v for k, v in xgb_params.items()
                          if k not in ('objective','n_jobs','random_state','verbosity')},
    "catboost_params":   catboost_params,
    "fold_scores_lgb":   [round(s, 4) for s in fold_scores_lgb],
    "fold_scores_xgb":   [round(s, 4) for s in fold_scores_xgb],
    "fold_scores_cat":   [round(s, 4) for s in fold_scores_cat],
    "use_log1p":         USE_LOG1P,
    "notes": "3-model ensemble: LGB (OOF encodings) + XGBoost (OOF encodings) + CatBoost (OTS). "
             "OOF geo×time_slot + gh5×time_slot + geo_day_trend.",
}
_log.append(_run)
json.dump(_log, open(_LOG_FILE, 'w'), indent=2)

print('Saved to results_log.json')
print(f"  LGB OOF R²     : {_run['lgb_oof_r2']}  weight={_run['lgb_weight']}")
print(f"  XGBoost OOF R² : {_run['xgb_oof_r2']}  weight={_run['xgb_weight']}")
print(f"  CatBoost OOF R²: {_run['cat_oof_r2']}  weight={_run['cat_weight']}")
print(f"  Ensemble OOF R²: {_run['oof_r2']}")
print(f"\nFill in lb_score after submitting submission_v2.csv")

Saved to results_log.json
  LGB OOF R²     : 0.954016  weight=0.2
  XGBoost OOF R² : 0.954338  weight=0.15
  CatBoost OOF R²: 0.95868  weight=0.65
  Ensemble OOF R²: 0.960645

Fill in lb_score after submitting submission_v2.csv


## Save submission

In [12]:
submission = pd.DataFrame({'Index': test_raw['Index'].values, 'demand': test_ensemble})
submission.to_csv('submission_v2.csv', index=False)

print(f'Saved {len(submission)} rows to submission_v2.csv')
print(f'demand range: [{submission.demand.min():.4f}, {submission.demand.max():.4f}]')
print(f'demand > 0 : {(submission.demand > 0).mean()*100:.1f}%')
submission.head(10)

Saved 41778 rows to submission_v2.csv
demand range: [0.0002, 1.0000]
demand > 0 : 100.0%


,Index,demand
0,0,0.054453
1,1,0.027107
2,2,0.024502
3,3,0.026269
4,4,0.077579
5,5,0.038243
6,6,0.043394
7,7,0.110285
8,8,0.043729
9,9,0.056340


## Residual Learning v2 — all 77k training rows

Baseline = demand_lag.fillna(geo_mean) for ALL rows (consistent with test inference). y_residual = demand − baseline. Gives 10× more training data than day-49-only approach.

In [20]:
t_resid2 = time.time()

# Consistent baseline for ALL training rows
baseline_train = train_raw['demand_lag'].fillna(
    train_raw['geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
).values
y_residual2 = y_train - baseline_train

# Test baseline (demand_lag already has geo fallback from cell-6 fix)
baseline_test = test_raw['demand_lag'].fillna(_global_demand_mean).values

print(f'All training rows: {len(y_residual2)}')
print(f'Residual range: [{y_residual2.min():.4f}, {y_residual2.max():.4f}]  mean={y_residual2.mean():.4f}  std={y_residual2.std():.4f}')

kf_r2 = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── LGB residual v2 ───────────────────────────────────────────────────────────
oof_lgb_r2 = np.zeros(n_train); test_lgb_r2 = np.zeros(len(test_raw))
lgb_r2_params = dict(objective='regression', metric='rmse', n_estimators=3000,
                     learning_rate=0.02, num_leaves=127, min_child_samples=20,
                     subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                     n_jobs=-1, verbose=-1, random_state=SEED)
print('Training LGB residual v2...')
for fold, (ti, vi) in enumerate(kf_r2.split(X_lgb_tr)):
    m = lgb.LGBMRegressor(**lgb_r2_params)
    m.fit(X_lgb_tr[ti], y_residual2[ti], eval_set=[(X_lgb_tr[vi], y_residual2[vi])],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(9999)])
    oof_lgb_r2[vi] = m.predict(X_lgb_tr[vi])
    test_lgb_r2   += m.predict(X_lgb_te) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual2[vi], oof_lgb_r2[vi]):.4f}  iter={m.best_iteration_}')
print(f'LGB residual v2 OOF R²={r2_score(y_residual2, oof_lgb_r2):.4f}')

# ── XGBoost residual v2 ───────────────────────────────────────────────────────
oof_xgb_r2 = np.zeros(n_train); test_xgb_r2 = np.zeros(len(test_raw))
xgb_r2_params = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.02,
                     max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                     reg_alpha=0.1, reg_lambda=0.1, early_stopping_rounds=150,
                     n_jobs=-1, random_state=SEED, verbosity=0)
print('Training XGBoost residual v2...')
for fold, (ti, vi) in enumerate(kf_r2.split(X_lgb_tr)):
    xgb_r2 = XGBRegressor(**xgb_r2_params)
    xgb_r2.fit(X_lgb_tr[ti], y_residual2[ti], eval_set=[(X_lgb_tr[vi], y_residual2[vi])], verbose=False)
    oof_xgb_r2[vi] = xgb_r2.predict(X_lgb_tr[vi])
    test_xgb_r2   += xgb_r2.predict(X_lgb_te) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual2[vi], oof_xgb_r2[vi]):.4f}  iter={xgb_r2.best_iteration}')
print(f'XGBoost residual v2 OOF R²={r2_score(y_residual2, oof_xgb_r2):.4f}')

# ── CatBoost residual v2 ──────────────────────────────────────────────────────
oof_cat_r2 = np.zeros(n_train); test_cat_r2 = np.zeros(len(test_raw))
cat_r2_params = dict(iterations=8000, learning_rate=0.03, depth=8,
                     loss_function='RMSE', random_seed=SEED, verbose=0, early_stopping_rounds=200)
print('Training CatBoost residual v2...')
for fold, (ti, vi) in enumerate(kf_r2.split(train_cat)):
    cb_r2 = CatBoostRegressor(**cat_r2_params)
    cb_r2.fit(train_cat.iloc[ti][CAT_FEAT_COLS], y_residual2[ti],
              cat_features=cat_feature_indices,
              eval_set=(train_cat.iloc[vi][CAT_FEAT_COLS], y_residual2[vi]),
              use_best_model=True)
    oof_cat_r2[vi] = cb_r2.predict(train_cat.iloc[vi][CAT_FEAT_COLS])
    test_cat_r2   += cb_r2.predict(test_cat[CAT_FEAT_COLS]) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual2[vi], oof_cat_r2[vi]):.4f}  iter={cb_r2.best_iteration_}')
print(f'CatBoost residual v2 OOF R²={r2_score(y_residual2, oof_cat_r2):.4f}')

# ── Blend search on residuals ─────────────────────────────────────────────────
best_r2_r2, best_wl_r2, best_wx_r2 = 0, 0.2, 0.1
for wl in np.arange(0.05, 0.70, 0.05):
    for wx in np.arange(0.05, 0.70, 0.05):
        if wl + wx >= 0.95: continue
        wc = 1.0 - wl - wx
        r2 = r2_score(y_residual2, wl*oof_lgb_r2 + wx*oof_xgb_r2 + wc*oof_cat_r2)
        if r2 > best_r2_r2:
            best_r2_r2, best_wl_r2, best_wx_r2 = r2, wl, wx
best_wc_r2 = 1.0 - best_wl_r2 - best_wx_r2
print(f'Best blend: LGB={best_wl_r2:.2f} XGB={best_wx_r2:.2f} CAT={best_wc_r2:.2f}  OOF R²={best_r2_r2:.4f}')

# Reconstruct demand
test_residual2 = best_wl_r2*test_lgb_r2 + best_wx_r2*test_xgb_r2 + best_wc_r2*test_cat_r2
test_reconstructed2 = np.clip(baseline_test + test_residual2, 0, 1)

# OOF reconstruction R² on demand
oof_reconstructed2 = np.clip(baseline_train + (best_wl_r2*oof_lgb_r2 + best_wx_r2*oof_xgb_r2 + best_wc_r2*oof_cat_r2), 0, 1)
print(f'OOF demand R²={r2_score(y_train, oof_reconstructed2):.4f}')
print(f'Test predictions: mean={test_reconstructed2.mean():.4f}  range=[{test_reconstructed2.min():.4f},{test_reconstructed2.max():.4f}]')

# Blend with direct best at optimal alpha
direct_best = pd.read_csv("submission_best_91780.csv")["demand"].values
for alpha in [0.15, 0.20, 0.22, 0.25, 0.30]:
    blend = np.clip((1-alpha)*direct_best + alpha*test_reconstructed2, 0, 1)
    delta = blend - direct_best
    print(f'  alpha={alpha:.2f}: shift std={delta.std():.5f}')

DATA_DIR = next((p for p in [Path('data'), Path('dataset')] if p.exists()), Path('data'))
test_df = pd.read_csv(DATA_DIR / 'test.csv')
pd.DataFrame({'Index': test_df['Index'].values, 'demand': test_reconstructed2}).to_csv('submission_residual_v2.csv', index=False)
print(f'Saved submission_residual_v2.csv')

# Also save the best blend (alpha=0.22)
blend_best = np.clip(0.78*direct_best + 0.22*test_reconstructed2, 0, 1)
pd.DataFrame({'Index': test_df['Index'].values, 'demand': blend_best}).to_csv('submission_blend_r2v2.csv', index=False)
print(f'Saved submission_blend_r2v2.csv (78% direct + 22% residual_v2)')
print(f'Runtime: {(time.time()-t_resid2)/60:.1f} min')


All training rows: 77299
Residual range: [-0.6276, 0.8360]  mean=0.0021  std=0.0810
Training LGB residual v2...
  Fold 1  R²=0.8602  iter=1709
  Fold 2  R²=0.8638  iter=2119
  Fold 3  R²=0.8697  iter=1836
  Fold 4  R²=0.8618  iter=2274
  Fold 5  R²=0.8645  iter=1505
LGB residual v2 OOF R²=0.8640
Training XGBoost residual v2...
  Fold 1  R²=0.8573  iter=2892
  Fold 2  R²=0.8630  iter=2992
  Fold 3  R²=0.8692  iter=2744
  Fold 4  R²=0.8631  iter=2996
  Fold 5  R²=0.8608  iter=2983
XGBoost residual v2 OOF R²=0.8627
Training CatBoost residual v2...
  Fold 1  R²=0.8621  iter=7999
  Fold 2  R²=0.8736  iter=7999
  Fold 3  R²=0.8732  iter=7999
  Fold 4  R²=0.8657  iter=7999
  Fold 5  R²=0.8676  iter=7970
CatBoost residual v2 OOF R²=0.8685
Best blend: LGB=0.30 XGB=0.15 CAT=0.55  OOF R²=0.8780
OOF demand R²=0.9606
Test predictions: mean=0.1240  range=[0.0000,1.0000]
  alpha=0.15: shift std=0.00332
  alpha=0.20: shift std=0.00443
  alpha=0.22: shift std=0.00487
  alpha=0.25: shift std=0.00554
  a

## Residual v1 Multi-seed — average residual predictions across seeds 42/123/999

Reduces variance in the residual component. Each seed run takes ~1 min on 7k rows.

In [23]:
t_rm = time.time()
RESID_SEEDS = [42, 123, 999]
test_resid_preds = np.zeros(len(test_raw))

d49_mask = train_raw['day'] == 49
y_d49    = train_raw.loc[d49_mask, 'demand'].values
lag_d49_filled = train_raw.loc[d49_mask, 'demand_lag'].fillna(
    train_raw.loc[d49_mask, 'geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
).values
y_residual_ms = y_d49 - lag_d49_filled
X_tr_d49 = X_lgb_tr[d49_mask.values]
train_cat_d49 = train_cat.loc[d49_mask].reset_index(drop=True)

for seed in RESID_SEEDS:
    print(f'\n--- Seed {seed} ---')
    kf_s = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    test_lgb_s = np.zeros(len(test_raw))
    test_xgb_s = np.zeros(len(test_raw))
    test_cat_s = np.zeros(len(test_raw))

    # LGB
    for fold, (ti, vi) in enumerate(kf_s.split(X_tr_d49)):
        m = lgb.LGBMRegressor(objective='regression', metric='rmse', n_estimators=3000,
                              learning_rate=0.02, num_leaves=63, min_child_samples=10,
                              subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                              n_jobs=-1, verbose=-1, random_state=seed)
        m.fit(X_tr_d49[ti], y_residual_ms[ti], eval_set=[(X_tr_d49[vi], y_residual_ms[vi])],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(9999)])
        test_lgb_s += m.predict(X_lgb_te) / N_FOLDS
    # XGB
    for fold, (ti, vi) in enumerate(kf_s.split(X_tr_d49)):
        xgb_s = XGBRegressor(objective='reg:squarederror', n_estimators=3000, learning_rate=0.02,
                             max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=0.1, early_stopping_rounds=150,
                             n_jobs=-1, random_state=seed, verbosity=0)
        xgb_s.fit(X_tr_d49[ti], y_residual_ms[ti], eval_set=[(X_tr_d49[vi], y_residual_ms[vi])], verbose=False)
        test_xgb_s += xgb_s.predict(X_lgb_te) / N_FOLDS
    # CatBoost
    for fold, (ti, vi) in enumerate(kf_s.split(train_cat_d49)):
        cb_s = CatBoostRegressor(iterations=8000, learning_rate=0.03, depth=7,
                                 loss_function='RMSE', random_seed=seed, verbose=0, early_stopping_rounds=200)
        cb_s.fit(train_cat_d49.iloc[ti][CAT_FEAT_COLS], y_residual_ms[ti],
                 cat_features=cat_feature_indices,
                 eval_set=(train_cat_d49.iloc[vi][CAT_FEAT_COLS], y_residual_ms[vi]),
                 use_best_model=True)
        test_cat_s += cb_s.predict(test_cat[CAT_FEAT_COLS]) / N_FOLDS

    # Use v1 weights: LGB=0.35, XGB=0.45, CAT=0.20
    seed_pred = 0.35*test_lgb_s + 0.45*test_xgb_s + 0.20*test_cat_s
    test_resid_preds += seed_pred / len(RESID_SEEDS)
    print(f'  Seed {seed} residual mean={seed_pred.mean():.4f}')

# Reconstruct demand
baseline_test_ms = test_raw['demand_lag'].fillna(_global_demand_mean).values
test_reconstructed_ms = np.clip(baseline_test_ms + test_resid_preds, 0, 1)

DATA_DIR = next((p for p in [Path('data'), Path('dataset')] if p.exists()), Path('data'))
test_df = pd.read_csv(DATA_DIR / 'test.csv')

# Save residual multi-seed and blend at alpha=0.22
pd.DataFrame({'Index': test_df['Index'].values, 'demand': test_reconstructed_ms}).to_csv('submission_residual_ms.csv', index=False)
direct_best = pd.read_csv('submission_best_91780.csv')['demand'].values
blend_ms = np.clip(0.78*direct_best + 0.22*test_reconstructed_ms, 0, 1)
pd.DataFrame({'Index': test_df['Index'].values, 'demand': blend_ms}).to_csv('submission_blend_residual_ms.csv', index=False)
delta = blend_ms - direct_best
print(f'\nResidual MS mean={test_reconstructed_ms.mean():.4f}  std={test_reconstructed_ms.std():.4f}')
print(f'Blend shift vs best: mean={delta.mean():.6f}  std={delta.std():.6f}')
print(f'Saved submission_residual_ms.csv and submission_blend_residual_ms.csv')
print(f'Runtime: {(time.time()-t_rm)/60:.1f} min')



--- Seed 42 ---
  Seed 42 residual mean=0.0271

--- Seed 123 ---
  Seed 123 residual mean=0.0267

--- Seed 999 ---
  Seed 999 residual mean=0.0271

Residual MS mean=0.1318  std=0.1829
Blend shift vs best: mean=0.001255  std=0.005852
Saved submission_residual_ms.csv and submission_blend_residual_ms.csv
Runtime: 3.8 min


## Residual Learning — predict (demand − demand_lag), add back at inference

Train only on day-49 rows (7,872 rows with valid demand_lag). Target = demand − demand_lag. At inference: pred = clip(demand_lag + model(features), 0, 1). Forces model to learn day49 deviation from day48 baseline rather than relearning the base signal.

In [21]:
t_resid = time.time()

# ── Filter to day-49 rows only (demand_lag is valid here) ─────────────────────
d49_mask = train_raw['day'] == 49
y_d49        = train_raw.loc[d49_mask, 'demand'].values
# Fill NaN demand_lag for day-49 rows (some geohashes missing from day-48)
lag_d49_series = train_raw.loc[d49_mask, 'demand_lag'].fillna(
    train_raw.loc[d49_mask, 'geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
)
lag_d49      = lag_d49_series.values
y_residual   = y_d49 - lag_d49   # target: deviation from day48 baseline

X_tr_d49 = X_lgb_tr[d49_mask.values]
train_cat_d49 = train_cat.loc[d49_mask].reset_index(drop=True)

print(f'Day-49 rows: {d49_mask.sum()}  residual range: [{y_residual.min():.4f}, {y_residual.max():.4f}]')
print(f'Residual mean={y_residual.mean():.4f}  std={y_residual.std():.4f}')

kf_r = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ── LGB residual ──────────────────────────────────────────────────────────────
oof_lgb_r = np.zeros(len(y_residual))
test_lgb_r = np.zeros(len(test_raw))
lgb_r_params = dict(objective='regression', metric='rmse', n_estimators=3000,
                    learning_rate=0.02, num_leaves=63, min_child_samples=10,
                    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                    n_jobs=-1, verbose=-1, random_state=SEED)
print('Training LGB residual...')
for fold, (ti, vi) in enumerate(kf_r.split(X_tr_d49)):
    m = lgb.LGBMRegressor(**lgb_r_params)
    m.fit(X_tr_d49[ti], y_residual[ti], eval_set=[(X_tr_d49[vi], y_residual[vi])],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(9999)])
    oof_lgb_r[vi] = m.predict(X_tr_d49[vi])
    test_lgb_r   += m.predict(X_lgb_te) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual[vi], oof_lgb_r[vi]):.4f}  iter={m.best_iteration_}')
print(f'LGB residual OOF R²={r2_score(y_residual, oof_lgb_r):.4f}')

# ── XGBoost residual ──────────────────────────────────────────────────────────
oof_xgb_r = np.zeros(len(y_residual))
test_xgb_r = np.zeros(len(test_raw))
xgb_r_params = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.02,
                    max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=0.1, early_stopping_rounds=150,
                    n_jobs=-1, random_state=SEED, verbosity=0)
print('Training XGBoost residual...')
for fold, (ti, vi) in enumerate(kf_r.split(X_tr_d49)):
    xgb_r = XGBRegressor(**xgb_r_params)
    xgb_r.fit(X_tr_d49[ti], y_residual[ti], eval_set=[(X_tr_d49[vi], y_residual[vi])], verbose=False)
    oof_xgb_r[vi] = xgb_r.predict(X_tr_d49[vi])
    test_xgb_r   += xgb_r.predict(X_lgb_te) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual[vi], oof_xgb_r[vi]):.4f}  iter={xgb_r.best_iteration}')
print(f'XGBoost residual OOF R²={r2_score(y_residual, oof_xgb_r):.4f}')

# ── CatBoost residual ─────────────────────────────────────────────────────────
oof_cat_r = np.zeros(len(y_residual))
test_cat_r = np.zeros(len(test_raw))
cat_r_params = dict(iterations=8000, learning_rate=0.03, depth=7,
                    loss_function='RMSE', random_seed=SEED, verbose=0, early_stopping_rounds=200)
print('Training CatBoost residual...')
for fold, (ti, vi) in enumerate(kf_r.split(train_cat_d49)):
    cb_r = CatBoostRegressor(**cat_r_params)
    cb_r.fit(train_cat_d49.iloc[ti][CAT_FEAT_COLS], y_residual[ti],
             cat_features=cat_feature_indices,
             eval_set=(train_cat_d49.iloc[vi][CAT_FEAT_COLS], y_residual[vi]),
             use_best_model=True)
    oof_cat_r[vi] = cb_r.predict(train_cat_d49.iloc[vi][CAT_FEAT_COLS])
    test_cat_r   += cb_r.predict(test_cat[CAT_FEAT_COLS]) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_residual[vi], oof_cat_r[vi]):.4f}  iter={cb_r.best_iteration_}')
print(f'CatBoost residual OOF R²={r2_score(y_residual, oof_cat_r):.4f}')

# ── Ensemble residuals → reconstruct demand ───────────────────────────────────
# OOF blend search on residuals
best_r2_r, best_wl_r, best_wx_r = 0, 0.2, 0.1
for wl in np.arange(0.05, 0.70, 0.05):
    for wx in np.arange(0.05, 0.70, 0.05):
        if wl + wx >= 0.95: continue
        wc = 1.0 - wl - wx
        blend = wl*oof_lgb_r + wx*oof_xgb_r + wc*oof_cat_r
        r2 = r2_score(y_residual, blend)
        if r2 > best_r2_r:
            best_r2_r, best_wl_r, best_wx_r = r2, wl, wx
best_wc_r = 1.0 - best_wl_r - best_wx_r
print(f'Best residual blend: LGB={best_wl_r:.2f} XGB={best_wx_r:.2f} CAT={best_wc_r:.2f}  OOF R²={best_r2_r:.4f}')

# Reconstruct: pred_demand = demand_lag + residual_pred
test_residual_pred = best_wl_r*test_lgb_r + best_wx_r*test_xgb_r + best_wc_r*test_cat_r
test_demand_lag    = test_raw['demand_lag'].values   # already has geo-mean fallback for missing
test_reconstructed = np.clip(test_demand_lag + test_residual_pred, 0, 1)

# Also check OOF reconstruction on day-49
oof_reconstructed = np.clip(lag_d49 + (best_wl_r*oof_lgb_r + best_wx_r*oof_xgb_r + best_wc_r*oof_cat_r), 0, 1)
print(f'OOF demand R²={r2_score(y_d49, oof_reconstructed):.4f}  (residual model reconstructed)')

print(f'Test predictions: mean={test_reconstructed.mean():.4f}  range=[{test_reconstructed.min():.4f},{test_reconstructed.max():.4f}]')

sub_r = pd.DataFrame({'Index': test_raw['Index'].values, 'demand': test_reconstructed})
sub_r.to_csv('submission_residual.csv', index=False)
print(f'Saved submission_residual.csv')
print(f'Residual model runtime: {(time.time()-t_resid)/60:.1f} min')


Day-49 rows: 7872  residual range: [-0.2779, 0.8360]
Residual mean=0.0405  std=0.0912
Training LGB residual...
  Fold 1  R²=0.8714  iter=686
  Fold 2  R²=0.8783  iter=772
  Fold 3  R²=0.8834  iter=673
  Fold 4  R²=0.8673  iter=794
  Fold 5  R²=0.8895  iter=572
LGB residual OOF R²=0.8786
Training XGBoost residual...
  Fold 1  R²=0.8697  iter=1518
  Fold 2  R²=0.8764  iter=1431
  Fold 3  R²=0.8865  iter=1258
  Fold 4  R²=0.8708  iter=1673
  Fold 5  R²=0.8932  iter=907
XGBoost residual OOF R²=0.8800
Training CatBoost residual...
  Fold 1  R²=0.8635  iter=4507
  Fold 2  R²=0.8571  iter=4097
  Fold 3  R²=0.8728  iter=2751
  Fold 4  R²=0.8450  iter=2380
  Fold 5  R²=0.8453  iter=4999
CatBoost residual OOF R²=0.8571
Best residual blend: LGB=0.35 XGB=0.45 CAT=0.20  OOF R²=0.8834
OOF demand R²=0.9543  (residual model reconstructed)
Test predictions: mean=0.1319  range=[0.0000,1.0000]
Saved submission_residual.csv
Residual model runtime: 1.5 min


## Confidence-Gated Pseudo-Labeling

Only pseudo-label test rows where all 6 seeds agree (per-row std < 0.005 → 91.3% of rows). Excludes the 8.7% of rows where the model is uncertain, preventing noisy labels from corrupting training. Uses 91.780 blend predictions as label values.

In [24]:
t_cgpl = time.time()
from pathlib import Path as _Path

# Load 6 seed predictions and compute per-row confidence
_seeds = [7, 13, 42, 77, 123, 999]
_seed_preds = np.stack([np.load(f'preds_seed{s}.npy') for s in _seeds], axis=0)
_per_row_std = _seed_preds.std(axis=0)
CONF_THRESH = 0.005
conf_mask = _per_row_std < CONF_THRESH
print(f'Confident rows (std < {CONF_THRESH}): {conf_mask.sum():,} / {len(conf_mask):,} ({conf_mask.mean()*100:.1f}%)')

# Use 91.780 blend predictions as pseudo-labels
_best_preds = pd.read_csv('submission_best_91780.csv')['demand'].values
pseudo_labels_cgpl = _best_preds[conf_mask]
print(f'Pseudo-label range: [{pseudo_labels_cgpl.min():.4f}, {pseudo_labels_cgpl.max():.4f}]')

# Build extended training set: real train + confident test rows
X_ext_cgpl = np.vstack([X_lgb_tr, X_lgb_te[conf_mask]])
y_ext_cgpl  = np.concatenate([y_train, pseudo_labels_cgpl])
cat_ext_cgpl = pd.concat(
    [train_cat[CAT_FEAT_COLS].reset_index(drop=True),
     test_cat[CAT_FEAT_COLS].reset_index(drop=True).loc[conf_mask]],
    ignore_index=True
)
print(f'Extended train: {len(y_ext_cgpl):,} rows ({n_train:,} real + {conf_mask.sum():,} confident pseudo)')

# Train LGB
lgb_cgpl = lgb.LGBMRegressor(objective='regression', n_estimators=2000, learning_rate=0.02,
                              num_leaves=127, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                              reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1, random_state=SEED)
lgb_cgpl.fit(X_ext_cgpl, y_ext_cgpl)
preds_lgb_cgpl = np.clip(lgb_cgpl.predict(X_lgb_te), 0, 1)
print(f'LGB done  mean={preds_lgb_cgpl.mean():.4f}')

# Train XGB
xgb_cgpl = XGBRegressor(objective='reg:squarederror', n_estimators=2000, learning_rate=0.02,
                         max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=SEED, verbosity=0)
xgb_cgpl.fit(X_ext_cgpl, y_ext_cgpl)
preds_xgb_cgpl = np.clip(xgb_cgpl.predict(X_lgb_te), 0, 1)
print(f'XGB done  mean={preds_xgb_cgpl.mean():.4f}')

# Train CatBoost
cb_cgpl = CatBoostRegressor(iterations=6000, learning_rate=0.03, depth=8,
                             loss_function='RMSE', random_seed=SEED, verbose=0)
cb_cgpl.fit(cat_ext_cgpl, y_ext_cgpl, cat_features=cat_feature_indices)
preds_cat_cgpl = np.clip(cb_cgpl.predict(test_cat[CAT_FEAT_COLS]), 0, 1)
print(f'CAT done  mean={preds_cat_cgpl.mean():.4f}')

# Blend with trained weights
direct_cgpl = np.clip(best_wl*preds_lgb_cgpl + best_wx*preds_xgb_cgpl + best_wc*preds_cat_cgpl, 0, 1)

# Also blend with residual v1 at alpha=0.22
_resid = pd.read_csv('submission_residual.csv')['demand'].values
final_cgpl = np.clip(0.78*direct_cgpl + 0.22*_resid, 0, 1)

_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_test_raw = pd.read_csv(_DATA_DIR / 'test.csv')
pd.DataFrame({'Index': _test_raw['Index'].values, 'demand': direct_cgpl}).to_csv('submission_cgpl_direct.csv', index=False)
pd.DataFrame({'Index': _test_raw['Index'].values, 'demand': final_cgpl}).to_csv('submission_cgpl_blend.csv', index=False)

delta = final_cgpl - _best_preds
print(f'\nDirect CGPL: mean={direct_cgpl.mean():.4f}')
print(f'Blend (78% CGPL + 22% residual): mean={final_cgpl.mean():.4f}')
print(f'Shift vs 91.780: mean={delta.mean():+.6f}  std={delta.std():.6f}')
print(f'Runtime: {(time.time()-t_cgpl)/60:.1f} min')


Confident rows (std < 0.005): 38,133 / 41,778 (91.3%)
Pseudo-label range: [0.0016, 1.0000]
Extended train: 115,432 rows (77,299 real + 38,133 confident pseudo)
LGB done  mean=0.1271
XGB done  mean=0.1268
CAT done  mean=0.1241

Direct CGPL: mean=0.1251
Blend (78% CGPL + 22% residual): mean=0.1266
Shift vs 91.780: mean=+0.000497  std=0.006334
Runtime: 2.2 min


## Residual Model Pseudo-Labeling

Pseudo-label the residual model: compute pseudo-residuals for test, retrain on day49 + pseudo-residuals. Consistent — everything stays in residual space. Then blend with direct best at alpha=0.22.

In [25]:
t_rp = time.time()
from pathlib import Path as _Path
_d49_mask = train_raw['day'] == 49
_y_d49    = train_raw.loc[_d49_mask, 'demand'].values
_lag_d49  = train_raw.loc[_d49_mask, 'demand_lag'].fillna(
    train_raw.loc[_d49_mask, 'geohash'].map(_geo_demand_mean).fillna(_global_demand_mean)
).values
_y_resid_d49 = _y_d49 - _lag_d49
_X_tr_d49    = X_lgb_tr[_d49_mask.values]
_tr_cat_d49  = train_cat.loc[_d49_mask].reset_index(drop=True)
_resid_preds  = pd.read_csv('submission_residual.csv')['demand'].values
_lag_test     = test_raw['demand_lag'].fillna(_global_demand_mean).values
_pseudo_resid = _resid_preds - _lag_test
print(f'Day-49: {len(_y_resid_d49)} rows  Pseudo-resid range=[{_pseudo_resid.min():.4f},{_pseudo_resid.max():.4f}]')
_X_ext_r  = np.vstack([_X_tr_d49, X_lgb_te])
_y_ext_r  = np.concatenate([_y_resid_d49, _pseudo_resid])
_cat_ext_r = pd.concat([_tr_cat_d49[CAT_FEAT_COLS].reset_index(drop=True),
                         test_cat[CAT_FEAT_COLS].reset_index(drop=True)], ignore_index=True)
print(f'Extended residual train: {len(_y_ext_r)} rows')
_lgb_rp = lgb.LGBMRegressor(objective='regression', n_estimators=2000, learning_rate=0.02,
                             num_leaves=63, min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1, random_state=SEED)
_lgb_rp.fit(_X_ext_r, _y_ext_r)
_t_lgb = _lgb_rp.predict(X_lgb_te)
print('LGB done')
_xgb_rp = XGBRegressor(objective='reg:squarederror', n_estimators=2000, learning_rate=0.02,
                        max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                        reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=SEED, verbosity=0)
_xgb_rp.fit(_X_ext_r, _y_ext_r)
_t_xgb = _xgb_rp.predict(X_lgb_te)
print('XGB done')
_cb_rp = CatBoostRegressor(iterations=6000, learning_rate=0.03, depth=7,
                            loss_function='RMSE', random_seed=SEED, verbose=0)
_cb_rp.fit(_cat_ext_r, _y_ext_r, cat_features=cat_feature_indices)
_t_cat = _cb_rp.predict(test_cat[CAT_FEAT_COLS])
print('CAT done')
_resid_pp = 0.35*_t_lgb + 0.45*_t_xgb + 0.20*_t_cat
_recon_rp = np.clip(_lag_test + _resid_pp, 0, 1)
_direct_best = pd.read_csv('submission_best_91780.csv')['demand'].values
_final_rp = np.clip(0.78*_direct_best + 0.22*_recon_rp, 0, 1)
_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _recon_rp}).to_csv('submission_resid_pseudo.csv', index=False)
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _final_rp}).to_csv('submission_resid_pseudo_blend.csv', index=False)
_delta = _final_rp - _direct_best
print(f'Blend shift vs 91.780: mean={_delta.mean():+.6f}  std={_delta.std():.6f}')
print(f'Saved submission_resid_pseudo_blend.csv  runtime: {(time.time()-t_rp)/60:.1f} min')


Day-49: 7872 rows  Pseudo-resid range=[-0.1843,0.6098]
Extended residual train: 49650 rows
LGB done
XGB done
CAT done
Blend shift vs 91.780: mean=+0.001258  std=0.005828
Saved submission_resid_pseudo_blend.csv  runtime: 0.9 min


## Day-48 Daytime-Only Model

Train on d48 rows at test timestamps (2:15-13:45) — exact distribution match to test. No demand_lag needed. Third orthogonal signal: direct model (77k rows, all timestamps) + residual model (7k day-49 rows) + daytime model (~34k d48 daytime rows).

In [26]:
t_dt = time.time()
from pathlib import Path as _Path

# Filter X_lgb_tr to d48 daytime rows (timestamps matching test)
_test_ts = set(test_raw['timestamp'].unique())
_dt_mask = (train_raw['day'] == 48) & (train_raw['timestamp'].isin(_test_ts))
_y_dt    = train_raw.loc[_dt_mask, 'demand'].values
_X_dt    = X_lgb_tr[_dt_mask.values]
_cat_dt  = train_cat.loc[_dt_mask].reset_index(drop=True)
print(f'Day-48 daytime rows: {_dt_mask.sum():,}  timestamps: {train_raw.loc[_dt_mask,"timestamp"].nunique()}')
print(f'Target range: [{_y_dt.min():.4f}, {_y_dt.max():.4f}]  mean={_y_dt.mean():.4f}')

_kf_dt = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# LGB
_oof_lgb_dt = np.zeros(len(_y_dt)); _test_lgb_dt = np.zeros(len(test_raw))
_lgb_dt_p = dict(objective='regression', metric='rmse', n_estimators=3000,
                 learning_rate=0.02, num_leaves=127, min_child_samples=20,
                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                 n_jobs=-1, verbose=-1, random_state=SEED)
print('Training LGB daytime...')
for fold, (ti, vi) in enumerate(_kf_dt.split(_X_dt)):
    m = lgb.LGBMRegressor(**_lgb_dt_p)
    m.fit(_X_dt[ti], _y_dt[ti], eval_set=[(_X_dt[vi], _y_dt[vi])],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(9999)])
    _oof_lgb_dt[vi] = np.clip(m.predict(_X_dt[vi]), 0, 1)
    _test_lgb_dt   += np.clip(m.predict(X_lgb_te), 0, 1) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(_y_dt[vi], _oof_lgb_dt[vi]):.4f}  iter={m.best_iteration_}')
print(f'LGB daytime OOF R²={r2_score(_y_dt, _oof_lgb_dt):.4f}')

# XGB
_oof_xgb_dt = np.zeros(len(_y_dt)); _test_xgb_dt = np.zeros(len(test_raw))
_xgb_dt_p = dict(objective='reg:squarederror', n_estimators=3000, learning_rate=0.02,
                 max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                 reg_alpha=0.1, reg_lambda=0.1, early_stopping_rounds=150,
                 n_jobs=-1, random_state=SEED, verbosity=0)
print('Training XGB daytime...')
for fold, (ti, vi) in enumerate(_kf_dt.split(_X_dt)):
    xgb_dt = XGBRegressor(**_xgb_dt_p)
    xgb_dt.fit(_X_dt[ti], _y_dt[ti], eval_set=[(_X_dt[vi], _y_dt[vi])], verbose=False)
    _oof_xgb_dt[vi] = np.clip(xgb_dt.predict(_X_dt[vi]), 0, 1)
    _test_xgb_dt   += np.clip(xgb_dt.predict(X_lgb_te), 0, 1) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(_y_dt[vi], _oof_xgb_dt[vi]):.4f}  iter={xgb_dt.best_iteration}')
print(f'XGB daytime OOF R²={r2_score(_y_dt, _oof_xgb_dt):.4f}')

# CatBoost
_oof_cat_dt = np.zeros(len(_y_dt)); _test_cat_dt = np.zeros(len(test_raw))
_cat_dt_p = dict(iterations=8000, learning_rate=0.03, depth=8,
                  loss_function='RMSE', random_seed=SEED, verbose=0, early_stopping_rounds=200)
print('Training CatBoost daytime...')
for fold, (ti, vi) in enumerate(_kf_dt.split(_cat_dt)):
    cb_dt = CatBoostRegressor(**_cat_dt_p)
    cb_dt.fit(_cat_dt.iloc[ti][CAT_FEAT_COLS], _y_dt[ti], cat_features=cat_feature_indices,
              eval_set=(_cat_dt.iloc[vi][CAT_FEAT_COLS], _y_dt[vi]), use_best_model=True)
    _oof_cat_dt[vi] = np.clip(cb_dt.predict(_cat_dt.iloc[vi][CAT_FEAT_COLS]), 0, 1)
    _test_cat_dt   += np.clip(cb_dt.predict(test_cat[CAT_FEAT_COLS]), 0, 1) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(_y_dt[vi], _oof_cat_dt[vi]):.4f}  iter={cb_dt.best_iteration_}')
print(f'CatBoost daytime OOF R²={r2_score(_y_dt, _oof_cat_dt):.4f}')

# Blend search
_best_r2_dt, _best_wl_dt, _best_wx_dt = 0, 0.2, 0.1
for wl in np.arange(0.05, 0.70, 0.05):
    for wx in np.arange(0.05, 0.70, 0.05):
        if wl + wx >= 0.95: continue
        wc = 1.0 - wl - wx
        r2 = r2_score(_y_dt, wl*_oof_lgb_dt + wx*_oof_xgb_dt + wc*_oof_cat_dt)
        if r2 > _best_r2_dt: _best_r2_dt, _best_wl_dt, _best_wx_dt = r2, wl, wx
_best_wc_dt = 1.0 - _best_wl_dt - _best_wx_dt
_daytime_pred = np.clip(_best_wl_dt*_test_lgb_dt + _best_wx_dt*_test_xgb_dt + _best_wc_dt*_test_cat_dt, 0, 1)
print(f'Daytime blend: LGB={_best_wl_dt:.2f} XGB={_best_wx_dt:.2f} CAT={_best_wc_dt:.2f}  OOF R²={_best_r2_dt:.4f}')
print(f'Daytime pred: mean={_daytime_pred.mean():.4f}  std={_daytime_pred.std():.4f}')

# Try blending with current best at various alphas
_direct_best = pd.read_csv('submission_best_91780.csv')['demand'].values
_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _daytime_pred}).to_csv('submission_daytime.csv', index=False)
for alpha in [0.10, 0.15, 0.20, 0.25]:
    blend = np.clip((1-alpha)*_direct_best + alpha*_daytime_pred, 0, 1)
    fname = f'submission_blend_dt{int(alpha*100):02d}.csv'
    pd.DataFrame({'Index': _test_df['Index'].values, 'demand': blend}).to_csv(fname, index=False)
    delta = blend - _direct_best
    print(f'  alpha={alpha:.2f}: shift std={delta.std():.5f} → {fname}')
print(f'Runtime: {(time.time()-t_dt)/60:.1f} min')


Day-48 daytime rows: 41,851  timestamps: 47
Target range: [0.0000, 1.0000]  mean=0.1059
Training LGB daytime...
  Fold 1  R²=0.9650  iter=554
  Fold 2  R²=0.9663  iter=906
  Fold 3  R²=0.9655  iter=523
  Fold 4  R²=0.9633  iter=698
  Fold 5  R²=0.9602  iter=1633
LGB daytime OOF R²=0.9640
Training XGB daytime...
  Fold 1  R²=0.9643  iter=1268
  Fold 2  R²=0.9653  iter=1771
  Fold 3  R²=0.9652  iter=1633
  Fold 4  R²=0.9639  iter=1739
  Fold 5  R²=0.9605  iter=2206
XGB daytime OOF R²=0.9638
Training CatBoost daytime...
  Fold 1  R²=0.9639  iter=4408
  Fold 2  R²=0.9650  iter=5370
  Fold 3  R²=0.9660  iter=4566
  Fold 4  R²=0.9653  iter=4435
  Fold 5  R²=0.9614  iter=5279
CatBoost daytime OOF R²=0.9643
Daytime blend: LGB=0.35 XGB=0.15 CAT=0.50  OOF R²=0.9669
Daytime pred: mean=0.1186  std=0.1666
  alpha=0.10: shift std=0.00297 → submission_blend_dt10.csv
  alpha=0.15: shift std=0.00445 → submission_blend_dt15.csv
  alpha=0.20: shift std=0.00593 → submission_blend_dt20.csv
  alpha=0.25: sh

## Daytime Model Pseudo-Labeling

Retrain daytime model on d48 daytime + 41k pseudo-labeled test rows (using 91.797 best as labels). Doubles training data while keeping distribution match. Then blend: 92% best91797 + 8% new daytime.

In [27]:
t_dtp = time.time()
from pathlib import Path as _Path

# d48 daytime rows
_test_ts_set = set(test_raw['timestamp'].unique())
_dt_mask2 = (train_raw['day'] == 48) & (train_raw['timestamp'].isin(_test_ts_set))
_y_dt2    = train_raw.loc[_dt_mask2, 'demand'].values
_X_dt2    = X_lgb_tr[_dt_mask2.values]
_cat_dt2  = train_cat.loc[_dt_mask2].reset_index(drop=True)

# Pseudo-labels from current best (91.797)
_best97 = pd.read_csv('submission_best_91797.csv')['demand'].values

# Extended training: d48 daytime + all test
_X_dtp   = np.vstack([_X_dt2, X_lgb_te])
_y_dtp   = np.concatenate([_y_dt2, _best97])
_cat_dtp = pd.concat([_cat_dt2[CAT_FEAT_COLS].reset_index(drop=True),
                       test_cat[CAT_FEAT_COLS].reset_index(drop=True)], ignore_index=True)
print(f'Extended daytime train: {len(_y_dtp):,} rows ({_dt_mask2.sum():,} d48 + {len(_best97):,} pseudo)')

# Train LGB
_lgb_dtp = lgb.LGBMRegressor(objective='regression', n_estimators=2000, learning_rate=0.02,
                              num_leaves=127, min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                              reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1, random_state=SEED)
_lgb_dtp.fit(_X_dtp, _y_dtp)
_p_lgb_dtp = np.clip(_lgb_dtp.predict(X_lgb_te), 0, 1)
print(f'LGB done  mean={_p_lgb_dtp.mean():.4f}')

# Train XGB
_xgb_dtp = XGBRegressor(objective='reg:squarederror', n_estimators=2000, learning_rate=0.02,
                         max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, random_state=SEED, verbosity=0)
_xgb_dtp.fit(_X_dtp, _y_dtp)
_p_xgb_dtp = np.clip(_xgb_dtp.predict(X_lgb_te), 0, 1)
print(f'XGB done  mean={_p_xgb_dtp.mean():.4f}')

# Train CatBoost
_cb_dtp = CatBoostRegressor(iterations=6000, learning_rate=0.03, depth=8,
                             loss_function='RMSE', random_seed=SEED, verbose=0)
_cb_dtp.fit(_cat_dtp, _y_dtp, cat_features=cat_feature_indices)
_p_cat_dtp = np.clip(_cb_dtp.predict(test_cat[CAT_FEAT_COLS]), 0, 1)
print(f'CAT done  mean={_p_cat_dtp.mean():.4f}')

# Blend with daytime OOF weights (from previous cell)
_daytime_pseudo_pred = np.clip(_best_wl_dt*_p_lgb_dtp + _best_wx_dt*_p_xgb_dtp + _best_wc_dt*_p_cat_dtp, 0, 1)

# Final: 92% of 91.797 best + 8% new daytime pseudo
_best97_arr = pd.read_csv('submission_best_91797.csv')['demand'].values
_final_dtp  = np.clip(0.92*_best97_arr + 0.08*_daytime_pseudo_pred, 0, 1)

_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _daytime_pseudo_pred}).to_csv('submission_daytime_pseudo.csv', index=False)
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _final_dtp}).to_csv('submission_daytime_pseudo_blend.csv', index=False)

_delta = _final_dtp - _best97_arr
print(f'Daytime pseudo pred: mean={_daytime_pseudo_pred.mean():.4f}')
print(f'Blend shift vs 91.797: mean={_delta.mean():+.6f}  std={_delta.std():.6f}')
print(f'Saved submission_daytime_pseudo_blend.csv  runtime: {(time.time()-t_dtp)/60:.1f} min')


Extended daytime train: 83,629 rows (41,851 d48 + 41,778 pseudo)
LGB done  mean=0.1255
XGB done  mean=0.1255
CAT done  mean=0.1256
Daytime pseudo pred: mean=0.1255
Blend shift vs 91.797: mean=+0.000002  std=0.000203
Saved submission_daytime_pseudo_blend.csv  runtime: 1.8 min


## Low-Rank Matrix Factorization

Build geohash×timestamp demand matrix from d48, apply SVD to extract smooth spatio-temporal structure, scale by day49/day48 per-geohash ratio. Completely different signal from tree models.

In [28]:
t_lr = time.time()
import numpy as np
from pathlib import Path as _Path
from scipy.linalg import svd as _svd

_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_train = pd.read_csv(_DATA_DIR / 'train.csv')
_test  = pd.read_csv(_DATA_DIR / 'test.csv')
_d48   = _train[_train['day']==48]
_d49   = _train[_train['day']==49]

# All geohashes (train + test) and test timestamps
_all_ghs = sorted(set(_train['geohash'].unique()) | set(_test['geohash'].unique()))
_test_ts  = sorted(_test['timestamp'].unique())
_gh_idx   = {gh: i for i, gh in enumerate(_all_ghs)}
_ts_idx   = {ts: i for i, ts in enumerate(_test_ts)}

# Build matrix M[geohash, timestamp] from d48 (test timestamps only)
_M = np.full((len(_all_ghs), len(_test_ts)), np.nan)
for _, row in _d48[_d48['timestamp'].isin(_test_ts)].iterrows():
    gi = _gh_idx.get(row['geohash'])
    ti = _ts_idx.get(row['timestamp'])
    if gi is not None and ti is not None:
        _M[gi, ti] = row['demand']

print(f'Matrix: {_M.shape}  missing: {np.isnan(_M).sum():,} / {_M.size:,} ({np.isnan(_M).mean()*100:.1f}%)')

# Impute missing with row mean (geohash mean), then column mean
_row_means = np.nanmean(_M, axis=1, keepdims=True)
_row_means[np.isnan(_row_means)] = np.nanmean(_M)
_M_imp = np.where(np.isnan(_M), _row_means, _M)

# Iterative SVD (2 iterations) for better imputation
K = 30
for _iter in range(3):
    _U, _S, _Vt = _svd(_M_imp, full_matrices=False)
    _M_recon = (_U[:, :K] * _S[:K]) @ _Vt[:K, :]
    _M_imp = np.where(np.isnan(_M), np.clip(_M_recon, 0, 1), _M)
print(f'SVD done  rank={K}  top singular values: {_S[:5].round(2)}')

# Final reconstruction
_U, _S, _Vt = _svd(_M_imp, full_matrices=False)
_M_recon = np.clip((_U[:, :K] * _S[:K]) @ _Vt[:K, :], 0, 1)

# Per-geohash day49/day48 scaling (early hours, Bayesian shrinkage)
_early_ts_set = set(_d49['timestamp'].unique())
_d49_em = _d49.groupby('geohash')['demand'].agg(['mean','count']).rename(columns={'mean':'m','count':'n'})
_d48_em = _d48[_d48['timestamp'].isin(_early_ts_set)].groupby('geohash')['demand'].mean()
_global_d49 = _d49['demand'].mean()
_global_d48e = _d48[_d48['timestamp'].isin(_early_ts_set)]['demand'].mean()
K_shrink = 20
_scales = {}
for gh in _all_ghs:
    if gh in _d49_em.index:
        n = _d49_em.loc[gh, 'n']; m = _d49_em.loc[gh, 'm']
        m_shrunk = (m*n + _global_d49*K_shrink) / (n + K_shrink)
        d48e = _d48_em.get(gh, _global_d48e)
        _scales[gh] = float(np.clip(m_shrunk / (d48e + 1e-4), 0.5, 2.0))
    else:
        _scales[gh] = float(np.clip(_global_d49 / (_global_d48e + 1e-4), 0.5, 2.0))

# Generate test predictions
_lr_preds = []
for _, row in _test.iterrows():
    gi = _gh_idx.get(row['geohash'])
    ti = _ts_idx.get(row['timestamp'])
    scale = _scales.get(row['geohash'], 1.0)
    if gi is not None and ti is not None:
        _lr_preds.append(float(np.clip(_M_recon[gi, ti] * scale, 0, 1)))
    else:
        _lr_preds.append(float(np.clip(_row_means[gi, 0] * scale if gi is not None else _global_d49, 0, 1)))
_lr_preds = np.array(_lr_preds)

print(f'LR preds: mean={_lr_preds.mean():.4f}  std={_lr_preds.std():.4f}  range=[{_lr_preds.min():.4f},{_lr_preds.max():.4f}]')

# Blend with current best
_best98 = pd.read_csv('submission_best_91798.csv')['demand'].values
pd.DataFrame({'Index': _test['Index'].values, 'demand': _lr_preds}).to_csv('submission_lowrank.csv', index=False)
for alpha in [0.05, 0.10, 0.15, 0.20]:
    blend = np.clip((1-alpha)*_best98 + alpha*_lr_preds, 0, 1)
    fname = f'submission_blend_lr{int(alpha*100):02d}.csv'
    pd.DataFrame({'Index': _test['Index'].values, 'demand': blend}).to_csv(fname, index=False)
    delta = blend - _best98
    print(f'  alpha={alpha:.2f}: shift mean={delta.mean():+.5f}  std={delta.std():.5f} → {fname}')
print(f'Runtime: {(time.time()-t_lr)/60:.1f} min')


Matrix: (1259, 47)  missing: 17,322 / 59,173 (29.3%)
SVD done  rank=30  top singular values: [37.66  5.18  2.8   2.14  1.85]
LR preds: mean=0.1411  std=0.1476  range=[0.0000,1.0000]
  alpha=0.05: shift mean=+0.00078  std=0.00464 → submission_blend_lr05.csv
  alpha=0.10: shift mean=+0.00156  std=0.00928 → submission_blend_lr10.csv
  alpha=0.15: shift mean=+0.00233  std=0.01392 → submission_blend_lr15.csv
  alpha=0.20: shift mean=+0.00311  std=0.01856 → submission_blend_lr20.csv
Runtime: 0.0 min


## Neural Network — sklearn MLP

MLP with 5-fold OOF on same features as LGB. Learns smooth nonlinear interactions vs tree piecewise functions. Blend at 5-15% with current best.

In [29]:
t_mlp = time.time()
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from pathlib import Path as _Path

# Scale features (MLP needs normalized inputs)
_scaler = StandardScaler()
_X_tr_sc = _scaler.fit_transform(X_lgb_tr)
_X_te_sc = _scaler.transform(X_lgb_te)

kf_mlp = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_mlp = np.zeros(n_train)
test_mlp = np.zeros(len(test_raw))

mlp_params = dict(
    hidden_layer_sizes=(256, 128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=1024,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=SEED,
    verbose=False
)

print('Training MLP (5-fold)...')
for fold, (ti, vi) in enumerate(kf_mlp.split(_X_tr_sc)):
    mlp = MLPRegressor(**mlp_params)
    mlp.fit(_X_tr_sc[ti], y_train[ti])
    oof_mlp[vi] = np.clip(mlp.predict(_X_tr_sc[vi]), 0, 1)
    test_mlp   += np.clip(mlp.predict(_X_te_sc), 0, 1) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_train[vi], oof_mlp[vi]):.4f}  iters={mlp.n_iter_}')

print(f'MLP OOF R²={r2_score(y_train, oof_mlp):.4f}')
print(f'MLP preds: mean={test_mlp.mean():.4f}  std={test_mlp.std():.4f}')

_best98 = pd.read_csv('submission_best_91798.csv')['demand'].values
_DATA_DIR = next((p for p in [_Path('data'), _Path('dataset')] if p.exists()), _Path('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')

pd.DataFrame({'Index': _test_df['Index'].values, 'demand': test_mlp}).to_csv('submission_mlp.csv', index=False)

for alpha in [0.05, 0.08, 0.10, 0.15, 0.20]:
    blend = np.clip((1-alpha)*_best98 + alpha*test_mlp, 0, 1)
    fname = f'submission_blend_mlp{int(alpha*100):02d}.csv'
    pd.DataFrame({'Index': _test_df['Index'].values, 'demand': blend}).to_csv(fname, index=False)
    delta = blend - _best98
    print(f'  alpha={alpha:.2f}: shift mean={delta.mean():+.5f}  std={delta.std():.5f} → {fname}')

print(f'Runtime: {(time.time()-t_mlp)/60:.1f} min')


Training MLP (5-fold)...
  Fold 1  R²=0.9455  iters=44
  Fold 2  R²=0.9448  iters=60
  Fold 3  R²=0.9488  iters=66
  Fold 4  R²=0.9414  iters=66
  Fold 5  R²=0.9446  iters=50
MLP OOF R²=0.9451
MLP preds: mean=0.1262  std=0.1688
  alpha=0.05: shift mean=+0.00003  std=0.00110 → submission_blend_mlp05.csv
  alpha=0.08: shift mean=+0.00006  std=0.00176 → submission_blend_mlp08.csv
  alpha=0.10: shift mean=+0.00007  std=0.00220 → submission_blend_mlp10.csv
  alpha=0.15: shift mean=+0.00010  std=0.00331 → submission_blend_mlp15.csv
  alpha=0.20: shift mean=+0.00014  std=0.00441 → submission_blend_mlp20.csv
Runtime: 1.4 min


## Stacked Neural Network (Level-2 Meta-Learner)

Uses OOF predictions from all component models as features: direct GBT blend, residual v1, daytime model. PyTorch MLP learns SAMPLE-SPECIFIC blend weights — much more powerful than fixed global weights from grid search.

In [31]:
t_stack = time.time()
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler as _SS
from pathlib import Path as _P

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Stacked NN  device={DEVICE}')

# Direct GBT blend OOF
_oof_direct = np.clip(best_wl*oof_lgb + best_wx*oof_xgb + best_wc*oof_cat, 0, 1)

# Build full-length residual OOF (only valid for d49 rows, fill rest with direct)
_d49_mask_s = (train_raw['day'] == 49).values
_full_oof_resid = _oof_direct.copy()
_full_oof_resid[_d49_mask_s] = np.clip(0.35*oof_lgb_r + 0.45*oof_xgb_r + 0.20*oof_cat_r, 0, 1)

# Build full-length daytime OOF (only valid for d48 daytime rows, fill rest with direct)
_test_ts_set_s = set(test_raw['timestamp'].unique())
_dt_mask_s = ((train_raw['day']==48) & train_raw['timestamp'].isin(_test_ts_set_s)).values
_full_oof_daytime = _oof_direct.copy()
_full_oof_daytime[_dt_mask_s] = np.clip(_best_wl_dt*_oof_lgb_dt + _best_wx_dt*_oof_xgb_dt + _best_wc_dt*_oof_cat_dt, 0, 1)

# Test predictions from each component
_test_direct   = pd.read_csv('submissions/best/submission_best_91580.csv')['demand'].values
_test_residual = pd.read_csv('submissions/experiments/submission_residual.csv')['demand'].values
_test_daytime  = pd.read_csv('submissions/experiments/submission_daytime.csv')['demand'].values

# Meta-feature matrix
_lag_tr = train_raw['demand_lag'].fillna(train_raw['demand_lag'].median()).values
_lag_te = test_raw['demand_lag'].fillna(test_raw['demand_lag'].median()).values

_meta_tr = np.column_stack([
    _oof_direct, _full_oof_resid, _full_oof_daytime,
    oof_lgb, oof_xgb, oof_cat,
    _lag_tr,
    train_raw['hour'].values, train_raw['hour_sin'].values, train_raw['hour_cos'].values,
    train_raw['geo_day_trend'].values, train_raw['neighbor_demand_mean'].values
]).astype(np.float32)

_meta_te = np.column_stack([
    _test_direct, _test_residual, _test_daytime,
    test_lgb_preds, test_xgb_preds, test_cat_preds,
    _lag_te,
    test_raw['hour'].values, test_raw['hour_sin'].values, test_raw['hour_cos'].values,
    test_raw['geo_day_trend'].values, test_raw['neighbor_demand_mean'].values
]).astype(np.float32)

_ss = _SS()
_meta_tr_sc = _ss.fit_transform(_meta_tr)
_meta_te_sc = _ss.transform(_meta_te)
print(f'Meta features: {_meta_tr_sc.shape[1]}  train={len(_meta_tr_sc)}')

class _MetaMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x).squeeze(1)

_kf_s = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
_oof_stack = np.zeros(n_train)
_test_stack = np.zeros(len(test_raw))

print('Training stacked MLP...')
for fold, (ti, vi) in enumerate(_kf_s.split(_meta_tr_sc)):
    torch.manual_seed(SEED + fold)
    _model = _MetaMLP(_meta_tr_sc.shape[1]).to(DEVICE)
    _opt = torch.optim.Adam(_model.parameters(), lr=5e-4, weight_decay=1e-4)
    _dl = DataLoader(TensorDataset(torch.FloatTensor(_meta_tr_sc[ti]), torch.FloatTensor(y_train[ti])),
                     batch_size=2048, shuffle=True)
    best_r2, best_state, no_imp = -1, None, 0
    for epoch in range(300):
        _model.train()
        for xb, yb in _dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _opt.zero_grad(); nn.MSELoss()(_model(xb), yb).backward(); _opt.step()
        _model.eval()
        with torch.no_grad():
            vp = np.clip(_model(torch.FloatTensor(_meta_tr_sc[vi]).to(DEVICE)).cpu().numpy(), 0, 1)
        vr2 = r2_score(y_train[vi], vp)
        if vr2 > best_r2: best_r2=vr2; best_state={k:v.clone() for k,v in _model.state_dict().items()}; no_imp=0
        else:
            no_imp += 1
            if no_imp >= 20: break
    _model.load_state_dict(best_state)
    _model.eval()
    with torch.no_grad():
        _oof_stack[vi] = np.clip(_model(torch.FloatTensor(_meta_tr_sc[vi]).to(DEVICE)).cpu().numpy(), 0, 1)
        _test_stack += np.clip(_model(torch.FloatTensor(_meta_te_sc).to(DEVICE)).cpu().numpy(), 0, 1) / N_FOLDS
    print(f'  Fold {fold+1}  R²={r2_score(y_train[vi], _oof_stack[vi]):.4f}')

print(f'Stacked OOF R²={r2_score(y_train, _oof_stack):.4f}')
print(f'Test: mean={_test_stack.mean():.4f}  std={_test_stack.std():.4f}')

_best98 = pd.read_csv('submissions/best/submission_best_91798.csv')['demand'].values
_DATA_DIR = next((p for p in [_P('data'), _P('dataset')] if p.exists()), _P('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')
pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _test_stack}).to_csv('submission_stack.csv', index=False)
for alpha in [0.05, 0.10, 0.15, 0.20]:
    blend = np.clip((1-alpha)*_best98 + alpha*_test_stack, 0, 1)
    fname = f'submission_blend_stack{int(alpha*100):02d}.csv'
    pd.DataFrame({'Index': _test_df['Index'].values, 'demand': blend}).to_csv(fname, index=False)
    print(f'  alpha={alpha:.2f}: shift mean={(blend-_best98).mean():+.5f}  std={(blend-_best98).std():.5f} → {fname}')
print(f'Runtime: {(time.time()-t_stack)/60:.1f} min')


Stacked NN  device=cuda
Meta features: 12  train=77299
Training stacked MLP...
  Fold 1  R²=0.9573
  Fold 2  R²=0.9586
  Fold 3  R²=0.9592
  Fold 4  R²=0.9551
  Fold 5  R²=0.9595
Stacked OOF R²=0.9580
Test: mean=0.1375  std=0.1885
  alpha=0.05: shift mean=+0.00060  std=0.00150 → submission_blend_stack05.csv
  alpha=0.10: shift mean=+0.00120  std=0.00300 → submission_blend_stack10.csv
  alpha=0.15: shift mean=+0.00180  std=0.00450 → submission_blend_stack15.csv
  alpha=0.20: shift mean=+0.00240  std=0.00600 → submission_blend_stack20.csv
Runtime: 3.6 min


## Multi-Seed Daytime Model

Run daytime model (d48 daytime rows) with seeds 42/123/999, average predictions. Reduces variance in the 8% daytime component. Blend with best at alpha=0.08.

In [32]:
t_dtms = time.time()
from pathlib import Path as _P

_test_ts_dms = set(test_raw['timestamp'].unique())
_dt_mask_ms  = (train_raw['day'] == 48) & train_raw['timestamp'].isin(_test_ts_dms)
_y_dt_ms     = train_raw.loc[_dt_mask_ms, 'demand'].values
_X_dt_ms     = X_lgb_tr[_dt_mask_ms.values]
_cat_dt_ms   = train_cat.loc[_dt_mask_ms].reset_index(drop=True)
print(f'Daytime rows: {_dt_mask_ms.sum():,}')

_SEEDS_DT = [42, 123, 999]
_test_daytime_ms = np.zeros(len(test_raw))

for _seed in _SEEDS_DT:
    print(f'\n--- Seed {_seed} ---')
    _kf_dms = KFold(n_splits=N_FOLDS, shuffle=True, random_state=_seed)
    _t_lgb_s = np.zeros(len(test_raw))
    _t_xgb_s = np.zeros(len(test_raw))
    _t_cat_s = np.zeros(len(test_raw))

    for fold, (ti, vi) in enumerate(_kf_dms.split(_X_dt_ms)):
        m = lgb.LGBMRegressor(objective='regression', metric='rmse', n_estimators=3000,
                              learning_rate=0.02, num_leaves=127, min_child_samples=20,
                              subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
                              n_jobs=-1, verbose=-1, random_state=_seed)
        m.fit(_X_dt_ms[ti], _y_dt_ms[ti], eval_set=[(_X_dt_ms[vi], _y_dt_ms[vi])],
              callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(9999)])
        _t_lgb_s += np.clip(m.predict(X_lgb_te), 0, 1) / N_FOLDS

    for fold, (ti, vi) in enumerate(_kf_dms.split(_X_dt_ms)):
        xgb_s = XGBRegressor(objective='reg:squarederror', n_estimators=3000, learning_rate=0.02,
                             max_depth=6, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=0.1, early_stopping_rounds=150,
                             n_jobs=-1, random_state=_seed, verbosity=0)
        xgb_s.fit(_X_dt_ms[ti], _y_dt_ms[ti], eval_set=[(_X_dt_ms[vi], _y_dt_ms[vi])], verbose=False)
        _t_xgb_s += np.clip(xgb_s.predict(X_lgb_te), 0, 1) / N_FOLDS

    for fold, (ti, vi) in enumerate(_kf_dms.split(_cat_dt_ms)):
        cb_s = CatBoostRegressor(iterations=8000, learning_rate=0.03, depth=8,
                                 loss_function='RMSE', random_seed=_seed, verbose=0, early_stopping_rounds=200)
        cb_s.fit(_cat_dt_ms.iloc[ti][CAT_FEAT_COLS], _y_dt_ms[ti], cat_features=cat_feature_indices,
                 eval_set=(_cat_dt_ms.iloc[vi][CAT_FEAT_COLS], _y_dt_ms[vi]), use_best_model=True)
        _t_cat_s += np.clip(cb_s.predict(test_cat[CAT_FEAT_COLS]), 0, 1) / N_FOLDS

    _seed_pred = np.clip(_best_wl_dt*_t_lgb_s + _best_wx_dt*_t_xgb_s + _best_wc_dt*_t_cat_s, 0, 1)
    _test_daytime_ms += _seed_pred / len(_SEEDS_DT)
    print(f'  Seed {_seed} daytime mean={_seed_pred.mean():.4f}')

print(f'\nDaytime multi-seed avg: mean={_test_daytime_ms.mean():.4f}  std={_test_daytime_ms.std():.4f}')

_best98 = pd.read_csv('submissions/best/submission_best_91798.csv')['demand'].values
_DATA_DIR = next((p for p in [_P('data'), _P('dataset')] if p.exists()), _P('data'))
_test_df = pd.read_csv(_DATA_DIR / 'test.csv')

pd.DataFrame({'Index': _test_df['Index'].values, 'demand': _test_daytime_ms}).to_csv('submission_daytime_ms.csv', index=False)

for alpha in [0.06, 0.08, 0.10, 0.12]:
    blend = np.clip((1-alpha)*_best98 + alpha*_test_daytime_ms, 0, 1)
    fname = f'submission_blend_dtms{int(alpha*100):02d}.csv'
    pd.DataFrame({'Index': _test_df['Index'].values, 'demand': blend}).to_csv(fname, index=False)
    delta = blend - _best98
    print(f'  alpha={alpha:.2f}: shift mean={delta.mean():+.5f}  std={delta.std():.5f} → {fname}')

print(f'Runtime: {(time.time()-t_dtms)/60:.1f} min')


Daytime rows: 41,851

--- Seed 42 ---
  Seed 42 daytime mean=0.1186

--- Seed 123 ---
  Seed 123 daytime mean=0.1187

--- Seed 999 ---
  Seed 999 daytime mean=0.1187

Daytime multi-seed avg: mean=0.1186  std=0.1666
  alpha=0.06: shift mean=-0.00041  std=0.00164 → submission_blend_dtms06.csv
  alpha=0.08: shift mean=-0.00055  std=0.00218 → submission_blend_dtms08.csv
  alpha=0.10: shift mean=-0.00069  std=0.00273 → submission_blend_dtms10.csv
  alpha=0.12: shift mean=-0.00083  std=0.00328 → submission_blend_dtms12.csv
Runtime: 13.6 min


## Pseudo-labeling — retrain on train + test predictions

Use current ensemble predictions as pseudo-labels for the test set. Retraining on train + pseudo-labeled test exposes the model to daytime demand patterns (2:15–13:45), directly attacking the distribution shift that inflates our OOF-vs-LB gap.

No CV here — we train once on the full extended dataset with fixed iteration counts derived from CV averages.

In [22]:
t_pseudo = time.time()

# Round 2: use previous round's predictions as pseudo-labels
_prev = pd.read_csv('submission_pseudo.csv')
pseudo_labels = _prev['demand'].values   # 91.507 predictions

X_ext_lgb = np.vstack([X_lgb_tr, X_lgb_te])
y_ext      = np.concatenate([y_train, pseudo_labels])

cat_ext_df = pd.concat(
    [train_cat[CAT_FEAT_COLS].reset_index(drop=True),
     test_cat[CAT_FEAT_COLS].reset_index(drop=True)],
    ignore_index=True
)

print(f'Extended train size: {len(y_ext)} rows  ({len(y_train)} real + {len(pseudo_labels)} pseudo)')
print(f'Pseudo-label range:  [{pseudo_labels.min():.4f}, {pseudo_labels.max():.4f}]')

LGB_ITERS = 2000
XGB_ITERS = 2000
CAT_ITERS = 6000

# ── LGB ───────────────────────────────────────────────────────────────────────
print('\nTraining LGB on extended data...')
lgb_p = lgb.LGBMRegressor(
    objective='regression', n_estimators=LGB_ITERS, learning_rate=0.02,
    num_leaves=127, min_child_samples=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, verbose=-1, random_state=SEED,
)
lgb_p.fit(X_ext_lgb, y_ext)
preds_lgb_p = np.clip(lgb_p.predict(X_lgb_te), 0, 1)
print(f'  done  mean={preds_lgb_p.mean():.4f}')

# ── XGBoost ───────────────────────────────────────────────────────────────────
print('Training XGBoost on extended data...')
xgb_p = XGBRegressor(
    objective='reg:squarederror', n_estimators=XGB_ITERS, learning_rate=0.02,
    max_depth=6, min_child_weight=20,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    n_jobs=-1, random_state=SEED, verbosity=0,
)
xgb_p.fit(X_ext_lgb, y_ext)
preds_xgb_p = np.clip(xgb_p.predict(X_lgb_te), 0, 1)
print(f'  done  mean={preds_xgb_p.mean():.4f}')

# ── CatBoost ──────────────────────────────────────────────────────────────────
print('Training CatBoost on extended data...')
cb_p = CatBoostRegressor(
    iterations=CAT_ITERS, learning_rate=0.03, depth=8,
    loss_function='RMSE', random_seed=SEED, verbose=0,
)
cb_p.fit(cat_ext_df, y_ext, cat_features=cat_feature_indices)
preds_cat_p = np.clip(cb_p.predict(test_cat[CAT_FEAT_COLS]), 0, 1)
print(f'  done  mean={preds_cat_p.mean():.4f}')

# ── Blend ─────────────────────────────────────────────────────────────────────
final_pseudo = np.clip(
    best_wl * preds_lgb_p + best_wx * preds_xgb_p + best_wc * preds_cat_p,
    0, 1
)

# Overwrite submission_pseudo.csv so next round just re-runs this same cell
sub_pseudo = pd.DataFrame({'Index': test_raw['Index'].values, 'demand': final_pseudo})
sub_pseudo.to_csv('submission_pseudo.csv', index=False)

delta = final_pseudo - pseudo_labels
print(f'\nPrediction shift vs input pseudo-labels:')
print(f'  mean delta : {delta.mean():+.5f}')
print(f'  std  delta : {delta.std():.5f}')
print(f'\nSaved submission_pseudo.csv  ({len(sub_pseudo)} rows)')
print(f'demand range: [{final_pseudo.min():.4f}, {final_pseudo.max():.4f}]')
print(f'Pseudo-label runtime: {(time.time()-t_pseudo)/60:.1f} min')
print('\nTo run another round: just re-run this cell (it reads submission_pseudo.csv each time)')

Extended train size: 119077 rows  (77299 real + 41778 pseudo)
Pseudo-label range:  [0.0009, 1.0000]

Training LGB on extended data...
  done  mean=0.1259
Training XGBoost on extended data...
  done  mean=0.1259
Training CatBoost on extended data...
  done  mean=0.1258

Prediction shift vs input pseudo-labels:
  mean delta : -0.00009
  std  delta : 0.00167

Saved submission_pseudo.csv  (41778 rows)
demand range: [0.0003, 1.0000]
Pseudo-label runtime: 2.7 min

To run another round: just re-run this cell (it reads submission_pseudo.csv each time)
